# grinpress v3: unattended teacher fine-tune + reflow distillation on an A100 🐾
**Runtime → GPU (A100) → Run all.** Re-running all cells is always safe:
- the whole pipeline (`run_v3.sh`) runs as a **background process** (`nohup setsid`), so a closed tab or a cell interrupt does not stop it;
- stages: **CRNN recognizer** (stage 0, Phase 2 on the GPU: scale channel + hard-class data, gate >= 98% on the 48 refs; `CRNN_FROM_REPO=1` uses the embedded weights instead) → teacher data pregen → **teacher fine-tune** (Phase 3, CRNN-validated on the 48-string grid, early stop > 95%) → **50k reflow pairs** → **student distill** (Phase 4) → **export + eval + bench** (Phase 5);
- every stage resumes: atomic local checkpoints + async **sharded** Drive mirror, chunk-level pair resume, CUDA-OOM batch halving (persisted), Drive retry/wait;
- `status.json` heartbeat, `run.log`, and the `CRNN_DONE` / `TEACHER_DONE` / `TRAIN_DONE` / `DONE` markers in `MyDrive/grinpress_v3/`. **Every Drive file is < 100 MB** (sharded ckpts, 1000-pair shards, fp16 weights, split fp32 ONNX);
- the monitor cell tails the log, remounts Drive when the FUSE mount wedges, and relaunches the run if the process died before `DONE`.

made with tail wags by the puppygirl CEO of Quantara (quantara.cv) :3

In [ ]:
# Mount Drive with retries (force_remount after a failure)
import os, time
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    for i in range(6):
        try:
            drive.mount('/content/drive', force_remount=i > 0, timeout_ms=180000)
            if os.path.isdir('/content/drive/MyDrive'):
                break
        except Exception as e:
            print(f'drive.mount attempt {i+1} failed: {e}')
            time.sleep(10 * (i + 1))
assert os.path.isdir('/content/drive/MyDrive'), 'Drive did not mount; re-run this cell'
DRIVE = '/content/drive/MyDrive/grinpress_v3'
os.makedirs(DRIVE, exist_ok=True)
os.makedirs('/content/grinpress_v3/artifacts', exist_ok=True)
print('Drive OK ->', DRIVE)

## Sources (written fresh every run; identical to the GitHub repo `Metrix187/grinpress-v3`)

In [ ]:
%%writefile /content/grinpress_v3/charset.py
"""Shared character set for v3 text: what the generator may print, what the CRNN recognises, and what
gets a dedicated tokenizer id (the tokenizer covers all printable ASCII; this is the printable-text subset).

' and " are excluded (they delimit the quoted span in captions); backslash and backtick are excluded too.
"""
LOWER = "abcdefghijklmnopqrstuvwxyz"
UPPER = LOWER.upper()
DIGITS = "0123456789"
PUNCT = " !?.,:;%$#&@*+-=~<>/()^_"
CHARSET = LOWER + UPPER + DIGITS + PUNCT
assert len(set(CHARSET)) == len(CHARSET)
BLANK = 0                                   # CTC blank
CHAR2ID = {c: i + 1 for i, c in enumerate(CHARSET)}
ID2CHAR = {i: c for c, i in CHAR2ID.items()}
N_CLASSES = len(CHARSET) + 1                # + blank


def encode(text):
    """text -> CRNN target ids (raises on chars outside CHARSET)."""
    return [CHAR2ID[c] for c in text]


def decode(ids):
    return "".join(ID2CHAR.get(int(i), "") for i in ids)


def ctc_greedy(seq):
    """argmax id sequence [T] -> string (collapse repeats, drop blanks)."""
    out, prev = [], None
    for k in seq:
        k = int(k)
        if k != prev and k != BLANK:
            out.append(k)
        prev = k
    return decode(out)


In [ ]:
%%writefile /content/grinpress_v3/conditioning.py
"""grinpress v3 character-level caption conditioning.

Changes vs v1/v2 (which used max_len 64 and silently truncated 21/48 eval prompts):
  * MAX_LEN = 128 (longest v2/v3 caption is 88 chars; headroom for free-form prompts).
  * Vocabulary: id = ASCII code point for every printable char (1..127), PAD = 0, and a DEDICATED UNK = 128
    (v1 mapped every non-ASCII char to '?', colliding with the real '?'). Every char of charset.CHARSET
    therefore has its own non-colliding id, identical to the v1 id (so v1 embedding rows warm-start 1:1).
  * Quoted-span features, computed from ids inside the module (so ONNX/index.html still only feed ids+mask):
      token_type: 0 = template text, 1 = the two quote delimiters of the FIRST '...' span, 2 = glyphs inside it
      glyph_idx : 0 outside the span, 1 + min(k, 15) for the k-th glyph inside (k = 0..15 relative index)
    Both are learned embeddings, ZERO-INITIALISED, so a v1 checkpoint loaded into this module behaves exactly like v1.
  * Unconditional (CFG) branch: the learned null sequence keeps its v1 length NULL_LEN = 64 and only those 64
    rows are attended to, so the uncond branch is also bit-for-bit v1 (padding it to 128 by repeating rows,
    as v2's loader did, changes the softmax).
"""
import math
import torch, torch.nn as nn

MAX_LEN = 128
NULL_LEN = 64
PAD, UNK = 0, 128
VOCAB = 130            # 0 pad, 1..127 ASCII, 128 unk, 129 reserved
QUOTE = ord("'")
N_GLYPH = 16           # relative glyph index 0..15 (clamped)


def tokenize(texts, max_len=MAX_LEN, strict=False):
    """List[str] -> (ids LongTensor [B,L], mask BoolTensor [B,L], True = real token).
    Non-ASCII / control chars -> UNK (128). strict=True raises on truncation or UNK instead."""
    ids = torch.zeros(len(texts), max_len, dtype=torch.long)
    for i, t in enumerate(texts):
        b = [ord(c) if 0 < ord(c) < 128 and (c.isprintable() or c == " ") else UNK for c in t]
        if strict and (len(b) > max_len or UNK in b):
            raise ValueError(f"caption needs truncation or has unknown chars: {t!r}")
        b = b[:max_len]
        if b:
            ids[i, :len(b)] = torch.tensor(b, dtype=torch.long)
    return ids, ids != PAD


def detokenize(ids):
    return ["".join("\ufffd" if int(k) == UNK else chr(int(k)) for k in row if int(k) != PAD) for row in ids]


def quote_features(ids):
    """ids [B,L] -> (token_type [B,L] in {0,1,2}, glyph_idx [B,L] in 0..16). Pure cumsum/compare ops (ONNX-friendly)."""
    q = (ids == QUOTE).long()
    c = torch.cumsum(q, 1)                                  # number of quotes seen up to and including this position
    inside = ((c == 1) & (q == 0)).long()                   # glyphs of the first quoted span
    delim = (q == 1) & (c <= 2)                             # its opening and closing quote
    k = torch.cumsum(inside, 1) - 1                         # relative glyph index within the span
    glyph = torch.where(inside == 1, 1 + torch.clamp(k, max=N_GLYPH - 1), torch.zeros_like(k))
    ttype = inside * 2 + delim.long()
    return ttype, glyph


def sinusoidal_1d(L, d):
    # PE[p,2i] = sin(p / 10000^(2i/d)), PE[p,2i+1] = cos(...). Positions 0..63 are identical to v1's 64-long table.
    pos = torch.arange(L).float()[:, None]
    div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
    pe = torch.zeros(L, d); pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
    return pe


class TextConditioner(nn.Module):
    """Embeds chars -> per-token sequence [B,L,D] + mask [B,L]. No pooling.

    CFG dropout: with prob p_uncond a sample's sequence is replaced by the learned null sequence
    (its first NULL_LEN positions marked valid, the rest masked)."""
    def __init__(self, dim=256, layers=2, heads=4, max_len=MAX_LEN, p_uncond=0.1, quote_feats=True):
        super().__init__()
        self.max_len, self.p_uncond, self.quote_feats = max_len, p_uncond, quote_feats
        self.emb = nn.Embedding(VOCAB, dim)
        self.register_buffer("pe", sinusoidal_1d(max_len, dim), persistent=False)
        if quote_feats:
            self.type_emb = nn.Embedding(3, dim)
            self.glyph_emb = nn.Embedding(N_GLYPH + 1, dim)
            nn.init.zeros_(self.type_emb.weight); nn.init.zeros_(self.glyph_emb.weight)
        layer = nn.TransformerEncoderLayer(dim, heads, dim * 4, dropout=0.0, batch_first=True,
                                           norm_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)
        self.null_seq = nn.Parameter(torch.randn(NULL_LEN, dim) * 0.02)

    def embed(self, ids):
        x = self.emb(ids) + self.pe[None, :ids.shape[1]]
        if self.quote_feats:
            tt, g = quote_features(ids)
            x = x + self.type_emb(tt) + self.glyph_emb(g)
        return x

    def null(self, B, L, dtype, device):
        n = min(L, NULL_LEN)
        x = torch.zeros(B, L, self.null_seq.shape[1], dtype=dtype, device=device)
        x[:, :n] = self.null_seq[:n].to(dtype)
        m = torch.zeros(B, L, dtype=torch.bool, device=device); m[:, :n] = True
        return x, m

    def forward(self, ids, mask, force_uncond=False):
        x = self.norm(self.encoder(self.embed(ids), src_key_padding_mask=~mask))
        B, L = ids.shape
        if force_uncond:
            drop = torch.ones(B, dtype=torch.bool, device=ids.device)
        elif self.training and self.p_uncond > 0:
            drop = torch.rand(B, device=ids.device) < self.p_uncond
        else:
            return x, mask
        nx, nm = self.null(B, L, x.dtype, x.device)
        x = torch.where(drop[:, None, None], nx, x)
        mask = torch.where(drop[:, None], nm, mask)
        return x, mask


In [ ]:
%%writefile /content/grinpress_v3/model.py
"""Pixel-space U-Net velocity model with self+cross attention to char tokens (v1 architecture; v3 conditioner)."""
import math, os
import torch, torch.nn as nn, torch.nn.functional as F
from conditioning import TextConditioner


def timestep_embedding(t, dim):
    # t in [0,1] scaled to [0,1000]; emb = [cos(t*w_i), sin(t*w_i)], w_i = 10000^(-i/half)
    half = dim // 2
    w = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device).float() / half)
    a = (t.float() * 1000.0)[:, None] * w[None]
    return torch.cat([torch.cos(a), torch.sin(a)], -1)


def gn(c): return nn.GroupNorm(32, c)


class ResBlock(nn.Module):
    def __init__(self, cin, cout, tdim, dropout=0.0):
        super().__init__()
        self.n1, self.c1 = gn(cin), nn.Conv2d(cin, cout, 3, padding=1)
        self.t = nn.Linear(tdim, cout * 2)
        self.n2, self.c2 = gn(cout), nn.Conv2d(cout, cout, 3, padding=1)
        self.drop = nn.Dropout(dropout)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
        nn.init.zeros_(self.c2.weight); nn.init.zeros_(self.c2.bias)

    def forward(self, x, temb):
        h = self.c1(F.silu(self.n1(x)))
        s, b = self.t(F.silu(temb))[:, :, None, None].chunk(2, 1)
        h = self.n2(h) * (1 + s) + b                                 # FiLM / AdaGN on timestep
        h = self.c2(self.drop(F.silu(h)))
        return self.skip(x) + h


class AttnBlock(nn.Module):
    """Self-attention over pixels, then cross-attention pixels->char tokens, then MLP."""
    def __init__(self, c, ctx_dim, heads=8):
        super().__init__()
        self.h = heads
        self.n1, self.qkv, self.o1 = gn(c), nn.Linear(c, 3 * c), nn.Linear(c, c)
        self.n2, self.q, self.kv, self.o2 = nn.LayerNorm(c), nn.Linear(c, c), nn.Linear(ctx_dim, 2 * c), nn.Linear(c, c)
        self.n3, self.mlp = nn.LayerNorm(c), nn.Sequential(nn.Linear(c, 4 * c), nn.GELU(), nn.Linear(4 * c, c))
        for o in (self.o1, self.o2, self.mlp[2]): nn.init.zeros_(o.weight); nn.init.zeros_(o.bias)

    def split(self, x):  # [B,N,C] -> [B,h,N,C/h]
        B, N, C = x.shape; return x.view(B, N, self.h, C // self.h).transpose(1, 2)

    def forward(self, x, ctx, mask):
        B, C, H, W = x.shape
        h = self.n1(x).flatten(2).transpose(1, 2)
        q, k, v = self.qkv(h).chunk(3, -1)
        a = F.scaled_dot_product_attention(self.split(q), self.split(k), self.split(v))  # softmax(QK^T/sqrt(d))V
        t = x.flatten(2).transpose(1, 2) + self.o1(a.transpose(1, 2).reshape(B, -1, C))
        k, v = self.kv(ctx).chunk(2, -1)
        am = mask[:, None, None, :]                                  # key padding mask: True = attend
        a = F.scaled_dot_product_attention(self.split(self.q(self.n2(t))), self.split(k), self.split(v), attn_mask=am)
        t = t + self.o2(a.transpose(1, 2).reshape(B, -1, C))
        t = t + self.mlp(self.n3(t))
        return t.transpose(1, 2).reshape(B, C, H, W)


class UNet(nn.Module):
    """Levels at 128px input: 128,64,32,16 (attn at the 32 and 16 levels, i.e. levels 2,3;
    at 256px input the same levels sit at 64/32)."""
    def __init__(self, ch=64, mults=(1, 2, 3, 4), attn_levels=(2, 3), nres=2, ctx_dim=256, tdim=512):
        super().__init__()
        self.tmlp = nn.Sequential(nn.Linear(ch, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.ch = ch
        self.inp = nn.Conv2d(3, ch, 3, padding=1)
        self.down, chans, c = nn.ModuleList(), [ch], ch
        for i, m in enumerate(mults):
            for _ in range(nres):
                blk = nn.ModuleList([ResBlock(c, ch * m, tdim)]); c = ch * m
                if i in attn_levels: blk.append(AttnBlock(c, ctx_dim))
                self.down.append(blk); chans.append(c)
            if i < len(mults) - 1:
                self.down.append(nn.ModuleList([nn.Conv2d(c, c, 3, stride=2, padding=1)])); chans.append(c)
        self.mid = nn.ModuleList([ResBlock(c, c, tdim), AttnBlock(c, ctx_dim), ResBlock(c, c, tdim)])
        self.up = nn.ModuleList()
        for i, m in reversed(list(enumerate(mults))):
            for _ in range(nres + 1):
                blk = nn.ModuleList([ResBlock(c + chans.pop(), ch * m, tdim)]); c = ch * m
                if i in attn_levels: blk.append(AttnBlock(c, ctx_dim))
                self.up.append(blk)
            if i > 0:
                self.up.append(nn.ModuleList([nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(c, c, 3, padding=1)]))
        self.out = nn.Sequential(gn(c), nn.SiLU(), nn.Conv2d(c, 3, 3, padding=1))
        nn.init.zeros_(self.out[2].weight); nn.init.zeros_(self.out[2].bias)

    def _run(self, blk, h, temb, ctx, mask):
        for m in blk:
            if isinstance(m, ResBlock): h = m(h, temb)
            elif isinstance(m, AttnBlock): h = m(h, ctx, mask)
            else: h = m(h)
        return h

    def forward(self, x, t, ctx, mask, temb_add=None):
        temb = self.tmlp(timestep_embedding(t, self.ch))
        if temb_add is not None:
            temb = temb + temb_add  # e.g. guidance-scale embedding (v2 student)
        h = self.inp(x); hs = [h]
        for blk in self.down:
            h = self._run(blk, h, temb, ctx, mask); hs.append(h)
        h = self._run(self.mid, h, temb, ctx, mask)
        for blk in self.up:
            if isinstance(blk[0], ResBlock): h = torch.cat([h, hs.pop()], 1)
            h = self._run(blk, h, temb, ctx, mask)
        return self.out(h)


class FlowModel(nn.Module):
    """Text conditioner + U-Net, trained jointly. forward returns predicted velocity."""
    def __init__(self, ctx_dim=256, **kw):
        super().__init__()
        self.cond = TextConditioner(dim=ctx_dim)
        self.unet = UNet(ctx_dim=ctx_dim, **kw)

    def forward(self, x, t, ids, mask, force_uncond=False):
        ctx, m = self.cond(ids, mask, force_uncond=force_uncond)
        return self.unet(x, t, ctx, m)


def count_params(m):
    return sum(p.numel() for p in m.parameters())


class StudentFlowModel(FlowModel):
    """v2 distilled student: 1 NFE per step, no CFG at inference.

    Optional guidance-scale embedding g(w) = MLP(sin_emb(w/10)) added to the time-embedding output.
    The last layer is zero-initialised, so at init the student equals the teacher's conditional branch.
    """
    def __init__(self, use_w_embed=True, w_default=5.0, ctx_dim=256, **kw):
        super().__init__(ctx_dim=ctx_dim, **kw)
        self.w_default = w_default
        self.w_mlp = None
        if use_w_embed:
            tdim = self.unet.tmlp[2].out_features
            self.w_mlp = nn.Sequential(nn.Linear(self.unet.ch, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
            nn.init.zeros_(self.w_mlp[2].weight); nn.init.zeros_(self.w_mlp[2].bias)

    def w_embed(self, w, B, device):
        if self.w_mlp is None:
            return None
        if w is None:
            w = self.w_default
        if not torch.is_tensor(w):
            w = torch.full((B,), float(w), device=device)
        return self.w_mlp(timestep_embedding(w / 10.0, self.unet.ch))  # w/10 so the *1000 scale lands in 0..800

    def forward(self, x, t, ids, mask, w=None):
        ctx, m = self.cond(ids, mask)
        return self.unet(x, t, ctx, m, temb_add=self.w_embed(w, x.shape[0], x.device))

    def forward_ctx(self, x, t, ctx, m, w=None):
        return self.unet(x, t, ctx, m, temb_add=self.w_embed(w, x.shape[0], x.device))


def load_state(path):
    """safetensors or .pt (with 'ema'/'model') -> flat fp32 state dict."""
    if str(path).endswith(".safetensors"):
        from safetensors.torch import load_file
        sd = load_file(str(path))
    else:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        sd = ck.get("ema", ck.get("model", ck))
    return {k: v.float() for k, v in sd.items()}


V3_NEW_KEYS = ("cond.type_emb", "cond.glyph_emb", "w_mlp")


def load_teacher_weights(module, path_or_sd):
    """Warm-start any v1/v2/v3 checkpoint into a v3 module.
    * v1 char embedding [128,D] -> v3 [130,D]: rows 0..127 copied 1:1 (same ASCII ids); UNK/reserved rows keep their init.
    * v1 null_seq [64,D] is used as-is (v3 attends to exactly NULL_LEN=64 null tokens).
    * v3-only params (quote type / glyph-index embeddings, zero-init; student w_mlp, zero-init) stay at init.
    Returns the list of keys left at init."""
    sd = load_state(path_or_sd) if isinstance(path_or_sd, (str, os.PathLike)) else dict(path_or_sd)
    own = module.state_dict()
    kept = []
    e = "cond.emb.weight"
    if e in sd and sd[e].shape != own[e].shape:
        w = own[e].clone(); n = min(sd[e].shape[0], w.shape[0]); w[:n] = sd[e][:n]; sd[e] = w
        kept.append(f"{e}[{n}:]")
    if "cond.null_seq" in sd and sd["cond.null_seq"].shape != own["cond.null_seq"].shape:
        old, L = sd["cond.null_seq"], own["cond.null_seq"].shape[0]  # v2-style padded null seq -> first L rows
        sd["cond.null_seq"] = old[:L] if old.shape[0] >= L else torch.cat([old, old[-1:].expand(L - old.shape[0], -1)])
    missing, unexpected = module.load_state_dict(sd, strict=False)
    assert not unexpected, unexpected
    assert all(k.startswith(V3_NEW_KEYS) for k in missing), missing
    return kept + list(missing)


PARAM_RANGE = (25e6, 40e6)


def build_model(verbose=True):
    m = FlowModel()
    n = sum(p.numel() for p in m.parameters())
    if verbose:
        nu = sum(p.numel() for p in m.unet.parameters())
        print(f"params: total {n/1e6:.2f}M (unet {nu/1e6:.2f}M, text {(n-nu)/1e6:.2f}M)")
    assert PARAM_RANGE[0] <= n <= PARAM_RANGE[1], f"param count {n} out of range"
    return m


if __name__ == "__main__":
    from conditioning import tokenize
    m = build_model()
    ids, mask = tokenize(["a red smiley face with 'HI' next to it"] * 2)
    print(m(torch.randn(2, 3, 128, 128), torch.rand(2), ids, mask).shape)


In [ ]:
%%writefile /content/grinpress_v3/generate_data.py
"""Synthetic 'vintage newspaper print' face+text dataset generator.

Each image: aged newsprint background, one halftone-filled face (smiley/sad/meh) on the
left, a short wood-type/serif text string on the right, distressed + ink-bled.
"""
import argparse, json, math, os, random, urllib.request
import numpy as np, cv2
from multiprocessing import Pool
from PIL import Image, ImageDraw, ImageFont

HERE = os.path.dirname(os.path.abspath(__file__))
FONT_DIR = os.path.join(HERE, "fonts")
FONT_URLS = [  # verified raw URLs (tried in order); at least one must succeed
    "https://raw.githubusercontent.com/google/fonts/main/ofl/oldstandardtt/OldStandard-Bold.ttf",
    "https://raw.githubusercontent.com/google/fonts/main/ofl/abrilfatface/AbrilFatface-Regular.ttf",
    "https://raw.githubusercontent.com/google/fonts/main/apache/ultra/Ultra-Regular.ttf",
    "https://raw.githubusercontent.com/google/fonts/main/ofl/rye/Rye-Regular.ttf",
    "https://raw.githubusercontent.com/google/fonts/main/ofl/oldstandardtt/OldStandard-Regular.ttf",
]

PALETTE = {
    "red": (200, 40, 40), "orange": (230, 120, 30), "yellow": (235, 195, 40),
    "green": (50, 150, 60), "teal": (30, 140, 140), "blue": (40, 80, 190),
    "navy": (25, 35, 110), "purple": (120, 50, 160), "pink": (230, 100, 160),
    "brown": (130, 80, 40), "gray": (120, 120, 120), "cyan": (40, 180, 220),
}
EXPRESSIONS = ["smiley", "sad", "meh"]
WORDS = ["NEWS", "HOT", "EXTRA", "WAR", "SALE", "HELLO", "OK", "YES", "NO", "WOW", "GOOD",
         "BAD", "MEH", "JOY", "FUN", "SAD", "DAILY", "TIMES", "POST", "FREE", "NEW", "BIG",
         "SOLD", "WIN", "LOVE", "HAPPY", "GLUM", "DEAL", "1923", "VOTE"]
LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"


def ensure_fonts():
    """Download fonts with urllib if missing; return list of usable TTF paths."""
    os.makedirs(FONT_DIR, exist_ok=True)
    paths = []
    for url in FONT_URLS:
        p = os.path.join(FONT_DIR, url.rsplit("/", 1)[1])
        if not os.path.exists(p) or os.path.getsize(p) < 10000:
            try:
                req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=30) as r, open(p + ".tmp", "wb") as f:
                    f.write(r.read())
                os.replace(p + ".tmp", p)
            except Exception as e:
                print(f"[font] failed {url}: {e}")
                continue
        try:
            ImageFont.truetype(p, 20)  # validate it parses
            paths.append(p)
        except Exception as e:
            print(f"[font] invalid {p}: {e}")
    if not paths:
        raise RuntimeError("Could not download any TTF font; check network access. "
                           "PIL default font is intentionally not used.")
    return paths


def random_text(rng):
    if rng.random() < 0.6:
        return rng.choice(WORDS)
    return "".join(rng.choice(LETTERS) for _ in range(rng.randint(1, 8)))


def newsprint(S, rng, fonts):
    """Aged paper: yellowed base, low-freq blotches, coarse fibers, faint background text."""
    nprng = np.random.default_rng(rng.randrange(2**31))
    base = np.array([228, 216, 186], np.float32) + nprng.uniform(-10, 10, 3)
    img = np.ones((S, S, 3), np.float32) * base
    low = cv2.resize(nprng.normal(0, 1, (8, 8)).astype(np.float32), (S, S), interpolation=cv2.INTER_CUBIC)
    img += low[..., None] * np.array([6, 7, 12])            # uneven yellowing (more in blue channel)
    yy, xx = np.mgrid[0:S, 0:S] / S - 0.5
    img -= (xx**2 + yy**2)[..., None] * np.array([30, 40, 70])  # browned edges vignette
    img += nprng.normal(0, 4, (S, S, 1))                     # grain
    fib = np.zeros((S, S), np.uint8)
    for _ in range(int(S * S / 400)):                        # coarse fibers: short dark/light strokes
        x, y = nprng.integers(0, S, 2); a = nprng.uniform(0, math.pi); L = nprng.uniform(S/60, S/18)
        cv2.line(fib, (int(x), int(y)), (int(x + L*math.cos(a)), int(y + L*math.sin(a))),
                 int(nprng.integers(30, 90)), 1, cv2.LINE_AA)
    img -= cv2.GaussianBlur(fib, (0, 0), 0.6)[..., None].astype(np.float32) * 0.35
    # faint random background text (bleed-through from the other side)
    bt = Image.new("L", (S, S), 0); d = ImageDraw.Draw(bt)
    f = ImageFont.truetype(rng.choice(fonts), max(6, S // 22))
    y = -rng.randint(0, S // 20)
    while y < S:
        line = " ".join("".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(rng.randint(2, 7)))
                        for _ in range(12))
        d.text((-rng.randint(0, S // 8), y), line, fill=255, font=f)
        y += int(S / 18)
    bt = np.asarray(bt, np.float32) / 255
    img -= bt[..., None] * rng.uniform(14, 28)
    return img


def halftone(S, cx, cy, r, rng):
    """Halftone dot screen: 45° grid, dot radius ~ sqrt(ink coverage) with a shading gradient."""
    m = np.zeros((S, S), np.uint8)
    cell = S / rng.uniform(26, 34)
    ang = math.radians(45 + rng.uniform(-5, 5)); ca, sa = math.cos(ang), math.sin(ang)
    lx, ly = -0.5, -0.6  # light direction for shading
    n = int(S / cell) + 4
    for i in range(-n, n):
        for j in range(-n, n):
            u, v = i * cell, j * cell
            x, y = cx + u*ca - v*sa, cy + u*sa + v*ca
            dx, dy = (x - cx) / r, (y - cy) / r
            if dx*dx + dy*dy > 1.05: continue
            cov = np.clip(0.55 - 0.3*(dx*lx + dy*ly), 0.15, 0.9)      # coverage in [0,1]
            rad = cell * math.sqrt(cov / math.pi)                    # area ∝ coverage
            cv2.circle(m, (int(x*4), int(y*4)), max(1, int(rad*4)), 255, -1, cv2.LINE_AA, shift=2)
    circ = np.zeros_like(m); cv2.circle(circ, (int(cx), int(cy)), int(r), 255, -1, cv2.LINE_AA)
    return (m.astype(np.float32) / 255) * (circ.astype(np.float32) / 255)


def face_ink(S, cx, cy, r, expr, rng):
    m = np.zeros((S, S), np.uint8); th = max(2, int(r * 0.09))
    cv2.circle(m, (int(cx), int(cy)), int(r), 255, th, cv2.LINE_AA)
    er = max(2, int(r * 0.11))
    for sx in (-1, 1):
        cv2.circle(m, (int(cx + sx*r*0.35), int(cy - r*0.28)), er, 255, -1, cv2.LINE_AA)
    mw, my = r * 0.5, cy + r * 0.32
    pts = []
    for k in range(25):
        u = -1 + 2 * k / 24
        bend = {"smiley": 0.28, "sad": -0.28, "meh": 0.0}[expr]
        y = my + bend * r * (1 - u*u) - bend * r * 0.5           # parabola: up-curve smile, down frown
        pts.append((cx + u * mw, y + (0 if expr != "sad" else r * 0.12)))
    cv2.polylines(m, [np.round(np.array(pts) * 4).astype(np.int32)], False, 255, th, cv2.LINE_AA, shift=2)
    return m.astype(np.float32) / 255


def text_ink(S, text, font_path, x0, x1, cy, rng):
    img = Image.new("L", (S, S), 0); d = ImageDraw.Draw(img)
    size = int(S * 0.32)
    while size > 6:  # largest size fitting the right-hand region
        f = ImageFont.truetype(font_path, size)
        l, t, rr, b = d.textbbox((0, 0), text, font=f)
        if rr - l <= (x1 - x0) and b - t <= S * 0.4: break
        size -= 2
    w, h = rr - l, b - t
    d.text((x0 + (x1 - x0 - w) / 2 - l, cy - h / 2 - t), text, fill=255, font=f)
    return np.asarray(img, np.float32) / 255


def distress(a, rng, S):
    """Ink bleed (blur+soft-threshold spread) and distressed edges (multiplicative noise holes)."""
    nprng = np.random.default_rng(rng.randrange(2**31))
    a = cv2.GaussianBlur(a, (0, 0), S / 400 * rng.uniform(0.8, 1.5))
    a = np.clip(a * 1.25, 0, 1)                             # bleed: spreads ink slightly outward
    noise = cv2.resize(nprng.random((S // 6, S // 6)).astype(np.float32), (S, S), interpolation=cv2.INTER_LINEAR)
    speck = nprng.random((S, S)) > 0.04
    holes = np.clip((noise - 0.12) * 4, 0, 1) * speck       # worn patches + speckle dropouts
    return a * holes * rng.uniform(0.85, 1.0)


def render(res, rng, fonts):
    S = res * 2  # supersample then downsample for clean AA
    color_name = rng.choice(list(PALETTE)); rgb = np.array(PALETTE[color_name], np.float32)
    expr = rng.choice(EXPRESSIONS); text = random_text(rng)
    paper = newsprint(S, rng, fonts)
    cx, cy = S * rng.uniform(0.22, 0.27), S * rng.uniform(0.46, 0.54); r = S * rng.uniform(0.16, 0.19)
    col_a = distress(halftone(S, cx, cy, r * 0.97, rng), rng, S)
    blk_a = distress(np.maximum(face_ink(S, cx, cy, r, expr, rng),
                                text_ink(S, text, rng.choice(fonts), S*0.48, S*0.96, S*rng.uniform(0.46, 0.54), rng)), rng, S)
    # subtractive ink model: paper * (1 - a*(1 - ink/255))
    img = paper * (1 - col_a[..., None] * (1 - rgb / 255))
    img = img * (1 - blk_a[..., None] * (1 - np.array([25, 22, 20]) / 255))
    img = cv2.resize(np.clip(img, 0, 255).astype(np.uint8), (res, res), interpolation=cv2.INTER_AREA)
    meta = {"color": {"name": color_name, "rgb": [round(c / 255, 4) for c in PALETTE[color_name]]},
            "expression": expr, "text": text,
            "caption": f"a {color_name} {expr} face with '{text}' next to it"}
    return Image.fromarray(img), meta


def _work(job):
    i, seed, res, out, fonts = job
    rng = random.Random(seed * 1_000_003 + i)  # deterministic per-sample
    img, meta = render(res, rng, fonts)
    rel = f"images/{i:06d}.png"; img.save(os.path.join(out, rel))
    return {"image_path": rel, **meta}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--num_samples", type=int, default=1000)
    ap.add_argument("--output_dir", default="data")
    ap.add_argument("--resolution", type=int, default=128, choices=[128, 256])
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--workers", type=int, default=os.cpu_count())
    a = ap.parse_args()
    fonts = ensure_fonts(); print("fonts:", [os.path.basename(f) for f in fonts])
    os.makedirs(os.path.join(a.output_dir, "images"), exist_ok=True)
    jobs = [(i, a.seed, a.resolution, a.output_dir, fonts) for i in range(a.num_samples)]
    with open(os.path.join(a.output_dir, "metadata.jsonl"), "w") as fo, Pool(max(1, a.workers)) as pool:
        for k, rec in enumerate(pool.imap(_work, jobs, chunksize=16)):  # imap keeps order
            fo.write(json.dumps(rec) + "\n")
            if (k + 1) % 1000 == 0: print(f"{k+1}/{a.num_samples}", flush=True)
    print(f"wrote {a.num_samples} samples to {a.output_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/grinpress_v3/generate_data_v2.py
"""grinpress v2/v3 dataset: 7 expressions, 3 layouts, 3 papers, text up to 14 chars.

v3 additions (render_v2 output for a given rng/seed is UNCHANGED, so the v2 eval reference renders are identical):
  * render_v2(..., aux=True) also returns {"text_mask": float32 [res,res] text-bbox mask, "text_bbox": (x0,y0,x1,y1)
    in output pixels, "target_ids": CRNN/CTC ids of the text (charset.py)}; optional ink_rgb for ink variety.
  * random_text_v3 / sample_spec_v3: the v3 curriculum (40% 9-14 chars, 30% mixed case + punctuation/emoticons,
    uniform over 7 expressions x 3 layouts x 3 papers). The 48+14 eval strings (eval_grid.py) and every word in
    them are held out of the v3 samplers.

Captions are backward compatible with v1: the base sentence is exactly the v1 template and the new
options are OPTIONAL suffixes that are omitted at their v1 defaults (face left, aged paper):
  a {color} {expression} face with '{text}' next to it[, face on the right | , text below][, on fresh paper | , on sepia paper]
The longest caption is 88 chars (> 64), so v2 conditioning uses max_len 96 (see V2_MAX_LEN / README).
"""
import argparse, json, math, os, random
from multiprocessing import Pool
import numpy as np, cv2
from PIL import Image
from generate_data import PALETTE, ensure_fonts, newsprint, halftone, text_ink, distress, WORDS, LETTERS

EXPRESSIONS_V2 = ["smiley", "sad", "meh", "wink", "angry", "dead", "uwu"]
LAYOUTS = {"left": "", "right": ", face on the right", "below": ", text below"}
PAPERS = {"aged": "", "fresh": ", on fresh paper", "sepia": ", on sepia paper"}
MAX_TEXT = 14
V2_MAX_LEN = 96


def caption_v2(color, expr, text, layout="left", paper="aged"):
    return f"a {color} {expr} face with '{text}' next to it{LAYOUTS[layout]}{PAPERS[paper]}"


def longest_caption():
    c = max(PALETTE, key=len); e = max(EXPRESSIONS_V2, key=len)
    return caption_v2(c, e, "W" * MAX_TEXT, max(LAYOUTS, key=lambda k: len(LAYOUTS[k])), max(PAPERS, key=lambda k: len(PAPERS[k])))


def random_text_v2(rng):
    r = rng.random()
    if r < 0.45:
        return rng.choice(WORDS)
    if r < 0.65:  # two words, still <= 14
        t = f"{rng.choice(WORDS)} {rng.choice(WORDS)}"
        return t if len(t) <= MAX_TEXT else rng.choice(WORDS)
    return "".join(rng.choice(LETTERS) for _ in range(rng.randint(1, MAX_TEXT)))


def _poly(m, pts, th):
    cv2.polylines(m, [np.round(np.array(pts) * 4).astype(np.int32)], False, 255, th, cv2.LINE_AA, shift=2)


def face_ink_v2(S, cx, cy, r, expr):
    """Outline + eyes + mouth for all 7 expressions (ink mask in [0,1])."""
    m = np.zeros((S, S), np.uint8); th = max(2, int(r * 0.09)); er = max(2, int(r * 0.11))
    cv2.circle(m, (int(cx), int(cy)), int(r), 255, th, cv2.LINE_AA)
    ex, ey = r * 0.35, cy - r * 0.28
    arc = lambda x0, y0, w, bend, n=21: [(x0 + w * (-1 + 2 * k / (n - 1)), y0 + bend * (1 - (-1 + 2 * k / (n - 1)) ** 2)) for k in range(n)]
    # eyes
    for sx in (-1, 1):
        x = cx + sx * ex
        if expr == "wink" and sx == 1:
            _poly(m, arc(x, ey + r * 0.05, er * 1.4, -r * 0.08), th)            # closed eye: ^ arc
        elif expr == "dead":
            d = er * 1.2
            cv2.line(m, (int(x - d), int(ey - d)), (int(x + d), int(ey + d)), 255, th, cv2.LINE_AA)
            cv2.line(m, (int(x - d), int(ey + d)), (int(x + d), int(ey - d)), 255, th, cv2.LINE_AA)
        elif expr == "uwu":
            _poly(m, arc(x, ey - r * 0.05, er * 1.4, r * 0.12), th)             # 'u' shaped closed eyes
        else:
            cv2.circle(m, (int(x), int(ey)), er, 255, -1, cv2.LINE_AA)
        if expr == "angry":                                                     # brows slanting down to the centre
            cv2.line(m, (int(x + sx * er * 1.8), int(ey - er * 2.6)), (int(x - sx * er * 1.6), int(ey - er * 1.4)), 255, th, cv2.LINE_AA)
    # mouth
    mw, my = r * 0.5, cy + r * 0.32
    if expr in ("smiley", "wink"):
        _poly(m, arc(cx, my - r * 0.14, mw, r * 0.28), th)
    elif expr in ("sad", "angry"):
        _poly(m, arc(cx, my + r * 0.26, mw * (0.8 if expr == "angry" else 1), -r * 0.28), th)
    elif expr == "meh":
        _poly(m, [(cx - mw, my), (cx + mw, my)], th)
    elif expr == "dead":                                                        # wobbly line
        _poly(m, [(cx - mw + 2 * mw * k / 30, my + r * 0.05 * math.sin(k / 30 * 4 * math.pi)) for k in range(31)], th)
    elif expr == "uwu":                                                         # 'w' mouth
        w = mw * 0.55
        _poly(m, arc(cx - w / 2, my, w / 2, r * 0.12) + arc(cx + w / 2, my, w / 2, r * 0.12), th)
    return m.astype(np.float32) / 255


def paper_v2(S, rng, fonts, kind):
    p = newsprint(S, rng, fonts)
    if kind == "fresh":     # brighter, less yellow: pull toward off-white
        p = 0.45 * p + 0.55 * np.array([246, 244, 236], np.float32)
    elif kind == "sepia":   # luminance -> sepia tone
        lum = p @ np.array([0.299, 0.587, 0.114], np.float32)
        p = np.clip(lum[..., None] * np.array([1.07, 0.88, 0.66], np.float32), 0, 255)
    return p


DEFAULT_INK = (25, 22, 20)


def render_v2(res, rng, fonts, color=None, expr=None, text=None, layout=None, paper=None, aux=False, ink_rgb=None):
    S = res * 2
    color = color or rng.choice(list(PALETTE)); rgb = np.array(PALETTE[color], np.float32)
    expr = expr or rng.choice(EXPRESSIONS_V2); text = text or random_text_v2(rng)
    layout = layout or rng.choices(list(LAYOUTS), weights=[0.5, 0.25, 0.25])[0]
    paper = paper or rng.choices(list(PAPERS), weights=[0.5, 0.25, 0.25])[0]
    bg = paper_v2(S, rng, fonts, paper)
    font = rng.choice(fonts)
    if layout == "below":
        r = S * rng.uniform(0.17, 0.2); cx, cy = S * rng.uniform(0.47, 0.53), S * rng.uniform(0.3, 0.34)
        tx0, tx1, tcy = S * 0.05, S * 0.95, S * rng.uniform(0.74, 0.78)
    else:
        r = S * rng.uniform(0.15, 0.18); cy = S * rng.uniform(0.46, 0.54)
        if layout == "left":
            cx = S * rng.uniform(0.2, 0.24); tx0, tx1 = S * 0.43, S * 0.97
        else:
            cx = S * rng.uniform(0.76, 0.8); tx0, tx1 = S * 0.03, S * 0.57
        tcy = S * rng.uniform(0.46, 0.54)
    col_a = distress(halftone(S, cx, cy, r * 0.97, rng), rng, S)
    t_ink = text_ink(S, text, font, tx0, tx1, tcy, rng)        # text_ink draws no random numbers
    blk_a = distress(np.maximum(face_ink_v2(S, cx, cy, r, expr), t_ink), rng, S)
    ink = np.array(ink_rgb if ink_rgb is not None else DEFAULT_INK)  # int array like v2 (float64 math, identical pixels)
    img = bg * (1 - col_a[..., None] * (1 - rgb / 255))
    img = img * (1 - blk_a[..., None] * (1 - ink / 255))
    img = cv2.resize(np.clip(img, 0, 255).astype(np.uint8), (res, res), interpolation=cv2.INTER_AREA)
    meta = {"color": {"name": color, "rgb": [round(c / 255, 4) for c in PALETTE[color]]}, "expression": expr,
            "text": text, "layout": layout, "paper": paper, "caption": caption_v2(color, expr, text, layout, paper)}
    if not aux:
        return Image.fromarray(img), meta
    return Image.fromarray(img), meta, text_aux(t_ink, res, text)


def text_aux(t_ink, res, text, pad=1):
    """Text-bbox mask at output resolution from the supersampled text ink, plus CTC target ids."""
    from charset import encode
    S = t_ink.shape[0]; sc = res / S
    ys, xs = np.nonzero(t_ink > 0.1)
    if len(xs) == 0:
        box = (0, 0, 0, 0)
    else:
        box = (max(0, int(np.floor(xs.min() * sc)) - pad), max(0, int(np.floor(ys.min() * sc)) - pad),
               min(res, int(np.ceil((xs.max() + 1) * sc)) + pad), min(res, int(np.ceil((ys.max() + 1) * sc)) + pad))
    m = np.zeros((res, res), np.float32); m[box[1]:box[3], box[0]:box[2]] = 1.0
    return {"text_mask": m, "text_bbox": box, "target_ids": encode(text)}


# ------------------------------------------------------------------ v3 text curriculum
_WORDS_V3 = """news extra daily times post press paper print ink sale deal free sold vote win love happy glum joy fun
good bad big new hot cold war peace yes nope maybe okay sure cool nice great super mega ultra tiny small large huge
dog dogs puppy pup pups doggo pupper hound kitty cat cats meow purr bunny fox wolf bear duck frog bee owl
boop bonk zoom nap naps treat treats snack snacks bone bones ball fetch walk walks sit stay paw paws tail tails
fluffy floof fuzzy sleepy hungry grumpy silly goofy derp smol chonk heck frick yay yippee hooray huzzah oops
hello howdy hiya bye ciao aloha thanks please sorry wait stop go run jump play read write call text mail email
morning evening night today tomorrow monday friday weekend summer winter spring autumn holiday party birthday
coffee tea cake pizza taco burger donut cookie apple banana cherry lemon mango peach berry candy sugar honey
city town street park beach river ocean forest garden house home office school store market bank cafe club
king queen boss chief captain hero star legend champ rookie team crew squad gang family friend buddy pal
money cash price cost rent bill tax gold silver stock trade market crash boom rally record update alert flash
weather sunny rainy storm snow windy foggy cloudy heat wave report story headline special edition issue volume
magnificent spectacular independence headquarters celebration announcement adventure wonderful fantastic
incredible remarkable delightful marvelous tremendous quarterly breakthrough everything something anywhere
programming computer internet keyboard telephone television radio camera rocket engine planet galaxy universe
""".split()


def _held_out():
    from eval_grid import HELD_OUT
    words = {w.lower() for s in HELD_OUT for w in s.split()}
    return HELD_OUT, words


def _word_pool():
    _, bad = _held_out()
    pool = [w for w in _WORDS_V3 if w.lower() not in bad]
    pool += [w.lower() for w in WORDS if w.lower() not in bad and w.lower() not in pool]
    return pool


EMOTICONS = [":)", ":(", ":D", ":P", ";)", ":O", "xD", "XD", "<3", "^_^", "^^", ">_<", "T_T", "-_-", "o_O", "O_o",
             "=)", ":-)", ":-(", "8)", "(:", "):", ":*", ">:(", "B)", "^w^", "~_~", "*_*", "@_@", ">.<", "o.o", ":v"]


def _case(rng, w, mode=None):
    mode = mode or rng.choice(["upper", "lower", "title", "random"])
    if mode == "upper": return w.upper()
    if mode == "lower": return w.lower()
    if mode == "title": return w[:1].upper() + w[1:].lower()
    return "".join(c.upper() if rng.random() < 0.4 else c.lower() for c in w)


def _rand_chars(rng, n, alphabet):
    s = "".join(rng.choice(alphabet) for _ in range(n))
    return s


def _clean(t):
    t = " ".join(t.split())
    return t[:MAX_TEXT].strip()


def _long_text(rng, pool):
    r = rng.random()
    if r < 0.45:                                  # 2-3 words, case-consistent (BREAKING NEWS-like)
        mode = rng.choices(["upper", "lower", "title", "random"], weights=[0.45, 0.3, 0.15, 0.1])[0]
        ws = [_case(rng, rng.choice(pool), mode) for _ in range(rng.randint(2, 3))]
        t = " ".join(ws)
        if rng.random() < 0.25: t += rng.choice(["!", "!!", "?", "?!", "...", "~", "."])
    elif r < 0.65:                                # one long word
        longw = [w for w in pool if len(w) >= 9] or pool
        t = _case(rng, rng.choice(longw))
    elif r < 0.85:                                # random chars
        alpha = rng.choice([UPPER_ALNUM, LOWER_ALNUM, MIXED_ALNUM, MIXED_ALNUM + "!?.-:%$#~ "])
        t = _rand_chars(rng, rng.randint(9, MAX_TEXT), alpha)
    else:                                         # number/word combos
        t = rng.choice([f"{rng.choice(pool).upper()} {rng.randint(0, 9999)}", f"{rng.randint(1, 99)} {rng.choice(pool)}s",
                        f"${rng.randint(1, 999)} {rng.choice(pool)}", f"{rng.randint(1, 12)}:{rng.randint(0, 59):02d} {rng.choice(pool)}"])
    t = _clean(t)
    if len(t) < 9:
        t = _clean(t + " " + _case(rng, rng.choice(pool), "upper" if t.isupper() else "lower"))
    return t


def _mixed_text(rng, pool):
    r = rng.random()
    if r < 0.2: return rng.choice(EMOTICONS)
    if r < 0.4:                                   # random case word (HeLLo-like) / camel case (GoodGirl-like)
        if rng.random() < 0.5: return _clean(_case(rng, rng.choice(pool), "random"))
        return _clean("".join(_case(rng, rng.choice(pool), "title") for _ in range(2)))
    if r < 0.5:                                   # brand-ish (iPhone/McDonald-like)
        w = rng.choice(pool)
        return _clean(rng.choice(["i", "e", "Mc", "Mac", "O'"[:1]]) + w[:1].upper() + w[1:].lower())
    if r < 0.75:                                  # word + punctuation
        w = _case(rng, rng.choice(pool))
        return _clean(rng.choice([w + "!", w + "?", w + "?!", w + "!!", w + "...", w + "~", w + ".", "#" + w, "@" + w,
                                  w + " :)", w + "-" + _case(rng, rng.choice(pool))]))
    # numbers + symbols ($5, 50%, 12:30, 3.14, #1-like)
    n = rng.randint(0, 999)
    return _clean(rng.choice([f"${n}", f"{n}%", f"#{n}", f"{rng.randint(0, 23)}:{rng.randint(0, 59):02d}",
                              f"{rng.randint(0, 99)}.{rng.randint(0, 99)}", f"{n}+", f"~{n}", f"{n}/{rng.randint(1, 99)}",
                              f"<{n}>", f"{n}!", f"{rng.randint(1, 9)}+{rng.randint(1, 9)}={rng.randint(2, 18)}",
                              f"x{n}", f"-{n}", f"{n}st", f"({n})", f"A{n}", f"${n}.{rng.randint(0, 99):02d}"]))


def _base_text(rng, pool):
    r = rng.random()
    if r < 0.35: return _clean(rng.choice(pool).upper())
    if r < 0.6: return _clean(_case(rng, rng.choice(pool), rng.choice(["lower", "title"])))
    if r < 0.8: return _rand_chars(rng, rng.randint(1, 8), rng.choice([UPPER_ALNUM, LOWER_ALNUM, MIXED_ALNUM]))
    if r < 0.9: return str(rng.randint(0, 10 ** rng.randint(1, 4) - 1)).zfill(rng.choice([1, 1, 2, 3]))
    return _rand_chars(rng, rng.randint(1, 4), PUNCT_RICH)


UPPER_ALNUM = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
LOWER_ALNUM = "abcdefghijklmnopqrstuvwxyz0123456789"
MIXED_ALNUM = UPPER_ALNUM + "abcdefghijklmnopqrstuvwxyz"
PUNCT_RICH = "!?.,:;%$#&@*+-=~<>/()^_" + "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
_POOL = None


def random_text_v3(rng, kind=None):
    """kind in {long, mixed, base}; default draws the v3 curriculum mix 40/30/30. Never returns a held-out eval string."""
    global _POOL
    if _POOL is None: _POOL = _word_pool()
    held, _ = _held_out()
    from charset import CHARSET
    kind = kind or rng.choices(["long", "mixed", "base"], weights=[0.4, 0.3, 0.3])[0]
    for _ in range(50):
        t = {"long": _long_text, "mixed": _mixed_text, "base": _base_text}[kind](rng, _POOL)
        t = "".join(c for c in t if c in CHARSET).strip()
        if t and len(t) <= MAX_TEXT and "  " not in t and t not in held:
            return t
    return "PUP"


def sample_spec_v3(rng, text_kind=None):
    """Uniform over 12 colors x 7 expressions x 3 layouts x 3 papers; text from the v3 curriculum."""
    return dict(color=rng.choice(list(PALETTE)), expr=rng.choice(EXPRESSIONS_V2), text=random_text_v3(rng, text_kind),
                layout=rng.choice(list(LAYOUTS)), paper=rng.choice(list(PAPERS)))


INK_VARIANTS = [(25, 22, 20), (10, 10, 12), (40, 30, 25), (30, 28, 45), (55, 35, 25), (20, 30, 30)]


def render_v3(res, rng, fonts, spec=None, ink_variety=False):
    """v3 training sample: (PIL image, meta, aux) for a curriculum spec (drawn if None)."""
    spec = spec or sample_spec_v3(rng)
    ink = rng.choice(INK_VARIANTS) if ink_variety and rng.random() < 0.5 else None
    return render_v2(res, rng, fonts, aux=True, ink_rgb=ink, **spec)


def _work(job):
    i, seed, res, out, fonts = job
    img, meta = render_v2(res, random.Random(seed * 1_000_003 + i), fonts)
    rel = f"images/{i:06d}.png"; img.save(os.path.join(out, rel))
    return {"image_path": rel, **meta}


def preview(path, res, fonts):
    """Row per expression (7) x columns: 3 layouts on aged paper + fresh + sepia."""
    rng = random.Random(0); cols = [("left", "aged"), ("right", "aged"), ("below", "aged"), ("left", "fresh"), ("left", "sepia")]
    colors = list(PALETTE)
    grid = Image.new("RGB", (res * len(cols), res * len(EXPRESSIONS_V2)))
    for r_, e in enumerate(EXPRESSIONS_V2):
        for c_, (lay, pap) in enumerate(cols):
            img, _ = render_v2(res, rng, fonts, color=colors[(r_ * 5 + c_) % 12], expr=e,
                               text=["WOOF", "BARK BARK", "HI", "GOOD GIRL", "UWU"][c_], layout=lay, paper=pap)
            grid.paste(img, (c_ * res, r_ * res))
    grid.save(path)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--num_samples", type=int, default=1000)
    ap.add_argument("--output_dir", default="data_v2")
    ap.add_argument("--resolution", type=int, default=128, choices=[128, 256])
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--workers", type=int, default=os.cpu_count())
    ap.add_argument("--preview", default="", help="only render a preview grid to this path")
    a = ap.parse_args()
    fonts = ensure_fonts()
    print(f"longest caption ({len(longest_caption())} chars): {longest_caption()}")
    if a.preview:
        preview(a.preview, a.resolution, fonts); print("saved", a.preview); return
    os.makedirs(os.path.join(a.output_dir, "images"), exist_ok=True)
    jobs = [(i, a.seed, a.resolution, a.output_dir, fonts) for i in range(a.num_samples)]
    with open(os.path.join(a.output_dir, "metadata.jsonl"), "w") as fo, Pool(max(1, a.workers)) as pool:
        for k, rec in enumerate(pool.imap(_work, jobs, chunksize=16)):
            fo.write(json.dumps(rec) + "\n")
    print(f"wrote {a.num_samples} samples to {a.output_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/grinpress_v3/eval_grid.py
"""The v3 evaluation grids: EXACTLY the v2 eval specs (grinpress-v2/eval/eval_samples.py + eval_compare.py).

grid48  : 48 strings, spec(i, s), noise / reference-render seed 1000+i (eval_ocr.json, samples_*step.png, reference_renders.png)
grid14  : 14 strings (PICK), expression EXPRS[i//2], layout left, paper aged, seed 2000+i (eval_compare.json)
These strings are HELD OUT from every v3 training text sampler (see generate_data_v2.random_text_v3).
"""
from generate_data import PALETTE
from generate_data_v2 import EXPRESSIONS_V2, LAYOUTS, PAPERS, caption_v2

STRINGS = [
    "hi", "ok", "no", "OK", "NO", "YES",
    "GOOD MORNING!!", "extraordinary", "BREAKING NEWS", "HELLO WORLD", "newspaper boy", "PUPPYGIRL CEO",
    "bork", "uwu", "nyaa~", "awoo", "woof woof", "BARK", "mlem", "owo", ">w<", ":3",
    "HeLLo", "Woof", "GoodGirl", "iPhone", "McDonald",
    "2026", "404", "1923", "3.14", "007", "12:30",
    "?!", "...", "wow!", "hey?", "a-ok", "#1", "50%", "$5",
    "NEWS", "HELLO", "SALE", "EXTRA", "LOVE", "VOTE", "HAPPY",
]
assert len(STRINGS) == 48 and len(set(STRINGS)) == 48 and all(len(s) <= 14 for s in STRINGS)
PICK = ["hi", "HELLO", "GOOD MORNING!!", "bork", "2026", "?!", "NEWS", "uwu", "extraordinary", "404", "LOVE", "nyaa~", "HeLLo", "VOTE"]
COLORS, LAYS, PAPS = list(PALETTE), list(LAYOUTS), list(PAPERS)
HELD_OUT = frozenset(STRINGS) | frozenset(PICK)


def spec48(i, s):
    return dict(text=s, expr=EXPRESSIONS_V2[i % 7], layout=LAYS[i % 3], paper=PAPS[(i // 3) % 3], color=COLORS[(i * 5) % 12])


def grid48():
    """[(spec, caption, seed)] for the 48-string grid."""
    out = []
    for i, s in enumerate(STRINGS):
        sp = spec48(i, s)
        out.append((sp, caption_v2(sp["color"], sp["expr"], sp["text"], sp["layout"], sp["paper"]), 1000 + i))
    return out


def grid14():
    out = []
    for i, s in enumerate(PICK):
        sp = dict(text=s, expr=EXPRESSIONS_V2[i // 2], layout="left", paper="aged", color=COLORS[(i * 5) % 12])
        out.append((sp, caption_v2(sp["color"], sp["expr"], sp["text"], sp["layout"], sp["paper"]), 2000 + i))
    return out


def all_eval_captions():
    return [c for _, c, _ in grid48()] + [c for _, c, _ in grid14()]


In [ ]:
%%writefile /content/grinpress_v3/gp_io.py
"""Hang-proof I/O + run-control helpers shared by every v3 GPU stage (ported from grinpress-v2 @49675d9).

* drive_retry      : each Drive FUSE attempt runs in a worker thread with a timeout; ENOENT/EIO/hangs are retried.
* atomic_save      : torch.save to tmp + os.replace (atomic on POSIX and Drive FUSE rename).
* save_sharded / load_sharded : big objects as <=95 MB parts + manifest (written LAST, atomically, with sha256) so
                     every Drive file stays under the 100 MB download cap; a half-written set is never "complete".
* Heartbeat        : status.json written from a daemon thread (skips if a previous write is still in flight).
* AsyncMirror      : copy local-SSD checkpoints to Drive in the background (sharded), keep the last N.
* OOM helpers      : is_oom / free_cuda; callers halve the (micro-)batch and persist it to a json.
"""
import glob, hashlib, io, json, os, shutil, threading, time
import torch

PART_BYTES = 95 * 1000 * 1000


def drive_retry(fn, what, tries=40, wait=15, attempt_timeout=120):
    for i in range(tries):
        box, err_box = [], []

        def _run():
            try:
                box.append(fn())
            except BaseException as e:  # noqa: BLE001
                err_box.append(e)

        th = threading.Thread(target=_run, daemon=True)
        th.start(); th.join(attempt_timeout)
        if th.is_alive():
            print(f"[io] {what} hung after {attempt_timeout}s; Drive FUSE stuck? retry {i+1}/{tries} in {wait}s", flush=True)
            if i == tries - 1:
                raise TimeoutError(f"{what} hung after {attempt_timeout}s x {tries}")
            time.sleep(wait); continue
        if err_box:
            err = err_box[0]
            if isinstance(err, RuntimeError) and "directory" not in str(err).lower() and "file" not in str(err).lower():
                raise err
            if not isinstance(err, (OSError, RuntimeError, TimeoutError)) or i == tries - 1:
                raise err
            print(f"[io] {what} failed ({err}); Drive hiccup? retry {i+1}/{tries} in {wait}s", flush=True)
            time.sleep(wait); continue
        return box[0] if box else None


def ensure_parent(path):
    parent = os.path.dirname(os.path.abspath(path))
    # only re-create the leaf dir when its parent is visible (mount alive): a dead mount never silently becomes local disk
    if not os.path.isdir(parent) and os.path.isdir(os.path.dirname(parent)):
        os.makedirs(parent, exist_ok=True)


def atomic_save(obj, path):
    def _do():
        ensure_parent(path)
        torch.save(obj, path + ".tmp"); os.replace(path + ".tmp", path)
    drive_retry(_do, f"save {os.path.basename(path)}")


def atomic_write_bytes(data, path):
    def _do():
        ensure_parent(path)
        with open(path + ".tmp", "wb") as f: f.write(data)
        os.replace(path + ".tmp", path)
    drive_retry(_do, f"write {os.path.basename(path)}")


def load_json(path, default):
    try:
        with open(path) as f: return json.load(f)
    except (OSError, ValueError):
        return default


def save_json(obj, path):
    atomic_write_bytes(json.dumps(obj, indent=1).encode(), path)


def save_sharded(obj, base):
    """Write obj as base.partNN (<=95 MB each) + base.manifest.json (last). Returns the part count."""
    buf = io.BytesIO(); torch.save(obj, buf); data = buf.getvalue()
    parts = [data[i:i + PART_BYTES] for i in range(0, len(data), PART_BYTES)] or [b""]
    for k, p in enumerate(parts):
        atomic_write_bytes(p, f"{base}.part{k:02d}")
    for stale in glob.glob(f"{base}.part*"):
        try:
            if int(stale.rsplit("part", 1)[1].split(".")[0]) >= len(parts): os.remove(stale)
        except (ValueError, OSError):
            pass
    save_json({"n_parts": len(parts), "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest(),
               "time": time.strftime("%F %T")}, f"{base}.manifest.json")
    return len(parts)


def sharded_exists(base):
    return os.path.exists(f"{base}.manifest.json")


def load_sharded(base, map_location="cpu"):
    man = drive_retry(lambda: json.load(open(f"{base}.manifest.json")), f"read {os.path.basename(base)} manifest")
    data = b"".join(drive_retry(lambda k=k: open(f"{base}.part{k:02d}", "rb").read(), f"read part {k}") for k in range(man["n_parts"]))
    if hashlib.sha256(data).hexdigest() != man["sha256"]:
        raise OSError(f"sha256 mismatch for {base}")
    return torch.load(io.BytesIO(data), map_location=map_location, weights_only=False)


def remove_sharded(base):
    for f in glob.glob(f"{base}.part*") + glob.glob(f"{base}.manifest.json"):
        try: os.remove(f)
        except OSError: pass


class Heartbeat:
    """status.json off the train thread; a wedged Drive FUSE must never stall training."""
    def __init__(self, path, every=60):
        self.path, self.every, self.last, self._writing = path, every, 0.0, False

    def __call__(self, force=False, **kw):
        if not self.path or (not force and time.time() - self.last < self.every) or self._writing:
            return
        self.last = time.time(); self._writing = True
        kw.update(time=time.strftime("%Y-%m-%d %H:%M:%S"), unix=int(time.time()))

        def _write():
            try:
                try:
                    ensure_parent(self.path)
                    with open(self.path + ".tmp", "w") as f: json.dump(kw, f)
                    os.replace(self.path + ".tmp", self.path)
                except OSError as err:
                    print(f"[hb] heartbeat write failed ({err}); will retry next beat", flush=True); self.last = 0.0
            finally:
                self._writing = False
        threading.Thread(target=_write, daemon=True).start()


class AsyncMirror:
    """Mirror local checkpoints to a Drive dir as sharded files in a daemon thread (one copy in flight; a request
    made while busy is queued as 'pending' and copied next, so the newest ckpt always reaches Drive).
    Call flush() before the process exits, otherwise the daemon thread dies with it."""
    def __init__(self, mirror_dir, keep=2):
        self.dir, self.keep, self.pending, self.th = mirror_dir, keep, None, None
        self.lock = threading.Lock()

    def __call__(self, local_path):
        if not self.dir:
            return
        with self.lock:
            self.pending = local_path
            if self.th is not None and self.th.is_alive():
                return
            self.th = threading.Thread(target=self._loop, daemon=True); self.th.start()

    def _loop(self):
        while True:
            with self.lock:
                p, self.pending = self.pending, None
            if p is None:
                return
            self._copy(p)

    def _copy(self, local_path):
        name = os.path.basename(local_path)
        try:
            os.makedirs(self.dir, exist_ok=True)
            obj = torch.load(local_path, map_location="cpu", weights_only=False)
            save_sharded(obj, os.path.join(self.dir, name))
            bases = sorted({m[:-len(".manifest.json")] for m in glob.glob(os.path.join(self.dir, "ckpt_*.pt.manifest.json"))})
            for old in bases[:-self.keep]: remove_sharded(old)
            print(f"[mirror] {name} -> {self.dir}", flush=True)
        except Exception as err:  # noqa: BLE001
            print(f"[mirror] {name} failed ({err}); local ckpt is still safe", flush=True)

    def flush(self, timeout=1800):
        th = self.th
        if th is not None:
            th.join(timeout)
            if th.is_alive(): print("[mirror] flush timed out; Drive copy still in flight", flush=True)


def list_ckpts(d):
    return sorted(glob.glob(os.path.join(d, "ckpt_*.pt")), key=lambda p: int(os.path.basename(p)[5:-3]))


def latest_resume_state(local_dir, mirror_dir=""):
    """Newest of (local ckpt_*.pt, mirrored sharded ckpt). Returns (state_dict_obj or None, source)."""
    loc = list_ckpts(local_dir)
    best_step, best = -1, None
    if loc:
        best_step, best = int(os.path.basename(loc[-1])[5:-3]), ("local", loc[-1])
    if mirror_dir and os.path.isdir(mirror_dir):
        for m in glob.glob(os.path.join(mirror_dir, "ckpt_*.pt.manifest.json")):
            s = int(os.path.basename(m)[5:-len(".pt.manifest.json")])
            if s > best_step: best_step, best = s, ("mirror", m[:-len(".manifest.json")])
    if best is None:
        return None, None
    for kind, p in ([best] + ([("local", loc[-2])] if len(loc) > 1 else [])):
        try:
            obj = torch.load(p, map_location="cpu", weights_only=False) if kind == "local" else load_sharded(p)
            return obj, p
        except Exception as err:  # noqa: BLE001 — a corrupt newest ckpt falls back to the previous one
            print(f"[resume] could not load {p} ({err}); trying older", flush=True)
    return None, None


_FAKE_OOM = {"left": int(os.environ.get("GP_FAKE_OOM", "0"))}


def fake_oom_hook(size):
    """Smoke-test hook: GP_FAKE_OOM=N raises a CUDA-OOM-like error on the first N calls with size > 1,
    so the halving/backoff paths can be exercised on a CPU box."""
    if _FAKE_OOM["left"] > 0 and size > 1:
        _FAKE_OOM["left"] -= 1
        raise RuntimeError("CUDA out of memory (simulated by GP_FAKE_OOM)")


def is_oom(e):
    return isinstance(e, torch.OutOfMemoryError) or "out of memory" in str(e).lower()


def free_cuda():
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def amp_ctx(dev, precision):
    if dev.type != "cuda" or precision == "fp32":
        return torch.autocast("cpu", enabled=False), None
    dt = torch.bfloat16 if precision == "bf16" or (precision == "auto" and torch.cuda.is_bf16_supported()) else torch.float16
    return torch.autocast("cuda", dtype=dt), (torch.amp.GradScaler("cuda") if dt == torch.float16 else None)


def write_marker(path, **kw):
    try:
        save_json(dict(kw, finished=time.strftime("%F %T")), path)
    except Exception as err:  # noqa: BLE001
        print(f"[io] marker {path} failed ({err})", flush=True)


In [ ]:
%%writefile /content/grinpress_v3/crnn.py
"""Tiny conv-CTC text recognizer ("CRNN" without the RNN) for grinpress OCR eval + as a frozen loss in Phases 3/4.

Why no LSTM: Phases 3/4 backprop THROUGH the frozen recognizer in eval mode, and cuDNN RNN backward refuses to run
in eval mode. The sequence model is therefore a stack of residual 1-D convolutions over the width axis
(receptive field covers the whole crop), which is plenty for single-line procedural text.

Input: a TEXT-BOX crop, grey, 40x200 (aspect 5:1, aspect-preserving: the box is padded and then grown to 5:1 around
its centre, so tiny 14-char strings get ~2.5x more vertical resolution than a fixed band crop). Boxes come from
  * the generator's GT text bbox (training data, Phase-3 CTC loss: "crop GT text box"), or
  * detect_boxes(): dark-ink bbox inside the layout's text band (eval on references / generated images, and the bbox
    stored with every Phase-4 reflow pair). The layout bands were measured from 600 renders (left x54-126/y40-85,
    right x3-74/y43-88, below x5-123/y74-121).
The crop is a differentiable function of the image (F.grid_sample), so the SAME code path serves eval, CRNN training
and the CTC loss on x0_hat during teacher/student training.
(v3 run 1 used a fixed 48x160 band crop instead: 79.2% exact on the 48 refs; see assets/crnn_train_run1_band.json.)
"""
import math, os
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from charset import N_CLASSES, BLANK, ctc_greedy, encode

H_IN, W_IN, RES = 40, 200, 128
ASPECT = W_IN / H_IN
LAYOUT_ID = {"left": 0, "right": 1, "below": 2}
BANDS = torch.tensor([[52, 36, 128, 92], [0, 38, 76, 94], [2, 70, 126, 124]], dtype=torch.float32)  # x0,y0,x1,y1 px


def to_gray(x):
    """[B,3,H,W] -> [B,1,H,W] (luma); single-channel input passes through."""
    if x.shape[1] == 1:
        return x
    w = torch.tensor([0.299, 0.587, 0.114], device=x.device, dtype=x.dtype).view(1, 3, 1, 1)
    return (x * w).sum(1, keepdim=True)


def band_boxes(layout_ids, jitter=None):
    """layout_ids [B] -> boxes [B,4] px; jitter [B,4] px offsets added to (x0,y0,x1,y1)."""
    b = BANDS.to(layout_ids.device)[layout_ids]
    return b if jitter is None else b + jitter


def crop_boxes(x, boxes, out_hw=(H_IN, W_IN)):
    """Differentiable per-sample crop+resize. x [B,C,H,W] (any range), boxes [B,4] in pixel coords."""
    B, C, H, W = x.shape
    x0, y0, x1, y1 = boxes.unbind(1)
    theta = torch.zeros(B, 2, 3, device=x.device, dtype=x.dtype)
    theta[:, 0, 0] = (x1 - x0) / W; theta[:, 0, 2] = (x0 + x1) / W - 1
    theta[:, 1, 1] = (y1 - y0) / H; theta[:, 1, 2] = (y0 + y1) / H - 1
    grid = F.affine_grid(theta, (B, C, *out_hw), align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)


def with_scale(crop, eb):
    """Append a constant 'absolute scale' channel (expanded box height / 64 px) -> [B,2,H,W]. The aspect-normalised
    crop throws away absolute glyph size, which is the only cue separating o/O, w/W, s/S, ... in all-x-height strings.
    1-channel CRNNs (in_ch=1) simply ignore channel 1, so every caller can always pass the 2-channel crop."""
    s = ((eb[:, 3] - eb[:, 1]) / 64.0).to(crop.dtype).view(-1, 1, 1, 1).expand(-1, 1, *crop.shape[-2:])
    return torch.cat([crop, s], 1)


def crop_band(x, layout_ids, jitter=None):
    """x [B,3|1,128,128] in [-1,1] -> grey crop of the whole layout band (aspect-normalised) [B,2,40,200]."""
    eb = expand_boxes(band_boxes(layout_ids, jitter), pad_frac=0.0, pad_px=0.0)
    return with_scale(crop_boxes(to_gray(x), eb), eb)


def expand_boxes(boxes, pad_frac=0.12, pad_px=2.0, aspect=None, min_h=8.0):
    """Pad a text bbox, then grow it to the CRNN aspect ratio around its centre (never shrinks)."""
    aspect = aspect or ASPECT
    x0, y0, x1, y1 = boxes.float().unbind(1)
    w, h = (x1 - x0).clamp_min(1), (y1 - y0).clamp_min(1)
    ph = torch.maximum(h * (1 + 2 * pad_frac) + 2 * pad_px, torch.full_like(h, min_h))
    pw = w * (1 + 2 * pad_frac) + 2 * pad_px
    H = torch.maximum(ph, pw / aspect); W = H * aspect
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    return torch.stack([cx - W / 2, cy - H / 2, cx + W / 2, cy + H / 2], 1)


def crop_text(x, boxes, jitter=None):
    """x [B,3|1,128,128] in [-1,1], boxes [B,4] tight text bboxes (px) -> CRNN input [B,2,40,200] (grey + scale)."""
    b = boxes.float().to(x.device)
    if jitter is not None:
        b = b + jitter.to(x.device)
    eb = expand_boxes(b)
    return with_scale(crop_boxes(to_gray(x), eb), eb)


FACE_MARGIN = 4   # ignore this many px at the face-side edge of a band (face outline / halftone can poke in)


@torch.no_grad()
def detect_boxes(x, layout_ids, rel=0.45, q=0.005, min_drop=35.0):
    """Dark-ink text bbox inside each sample's layout band. x [B,3|1,128,128] in [-1,1] -> float boxes [B,4] (px).
    ink = luma < median - max(min_drop, rel * (median - darkest)), i.e. adaptive to faint tiny text;
    bbox from the q / 1-q coordinate quantiles; falls back to the band."""
    g = (to_gray(x.float()) + 1) * 127.5
    out = []
    for i in range(len(g)):
        lid = int(layout_ids[i]); bx0, by0, bx1, by1 = [int(v) for v in BANDS[lid].tolist()]
        if lid == 0: bx0 += FACE_MARGIN
        elif lid == 1: bx1 -= FACE_MARGIN
        else: by0 += FACE_MARGIN
        reg = g[i, 0, by0:by1, bx0:bx1]
        med = reg.median(); dark = torch.quantile(reg.flatten(), 0.002)
        ys, xs = torch.nonzero(reg < med - torch.clamp(rel * (med - dark), min=min_drop), as_tuple=True)
        if len(xs) < 6:
            out.append([bx0, by0, bx1, by1]); continue
        xs, ys = xs.float(), ys.float()
        out.append([bx0 + float(torch.quantile(xs, q)), by0 + float(torch.quantile(ys, q)),
                    bx0 + float(torch.quantile(xs, 1 - q)) + 1, by0 + float(torch.quantile(ys, 1 - q)) + 1])
    return torch.tensor(out, dtype=torch.float32, device=x.device)


def cbr(cin, cout, k=3, s=1, p=1):
    return nn.Sequential(nn.Conv2d(cin, cout, k, s, p, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class Res1d(nn.Module):
    def __init__(self, c, k=5):
        super().__init__()
        self.c1, self.b1 = nn.Conv1d(c, c, k, padding=k // 2, bias=False), nn.BatchNorm1d(c)
        self.c2, self.b2 = nn.Conv1d(c, c, k, padding=k // 2, bias=False), nn.BatchNorm1d(c)

    def forward(self, x):
        return F.relu(x + self.b2(self.c2(F.relu(self.b1(self.c1(x))))))


class CRNN(nn.Module):
    """[B,1|2,40,200] grey crop (+ optional scale channel) -> log-probs [T=50,B,N_CLASSES] (CTC layout)."""
    def __init__(self, w=(24, 48, 96, 128, 160), seq_layers=3, in_ch=1):
        super().__init__()
        c1, c2, c3, c4, cs = w
        self.in_ch = in_ch
        self.cfg = dict(w=list(w), seq_layers=seq_layers, in_ch=in_ch)
        self.cnn = nn.Sequential(
            cbr(in_ch, c1), nn.MaxPool2d(2),                       # 20x100
            cbr(c1, c2), nn.MaxPool2d(2),                      # 10x50
            cbr(c2, c3), cbr(c3, c3), nn.MaxPool2d((2, 1)),    # 5x50
            cbr(c3, c4), cbr(c4, c4), nn.MaxPool2d((2, 1), ceil_mode=True),  # 3x50
            cbr(c4, cs, k=(3, 1), p=0))                        # 1x50
        self.seq = nn.Sequential(*[Res1d(cs) for _ in range(seq_layers)])
        self.head = nn.Conv1d(cs, N_CLASSES, 1)

    def forward(self, crop):
        # per-crop standardisation: robust to paper tone / ink darkness / generated-image contrast
        g = crop[:, :1]
        mu = g.mean((2, 3), keepdim=True); sd = g.std((2, 3), keepdim=True) + 1e-3
        g = (g - mu) / sd
        if self.in_ch == 2:
            sc = crop[:, 1:2] if crop.shape[1] > 1 else torch.zeros_like(g)
            g = torch.cat([g, sc], 1)
        h = self.cnn(g).squeeze(2)              # [B,C,50]
        return self.head(self.seq(h)).permute(2, 0, 1).log_softmax(-1)   # [T,B,K]


def ctc_loss(logp, texts_or_ids, reduction="mean"):
    """CTC on log-probs [T,B,K]; targets as strings or id lists. Mean over batch of per-sample loss / target length."""
    ids = [encode(t) if isinstance(t, str) else list(t) for t in texts_or_ids]
    tgt = torch.tensor([i for s in ids for i in s], dtype=torch.long, device=logp.device)
    tl = torch.tensor([len(s) for s in ids], dtype=torch.long, device=logp.device)
    il = torch.full((logp.shape[1],), logp.shape[0], dtype=torch.long, device=logp.device)
    return F.ctc_loss(logp.float(), tgt, il, tl, blank=BLANK, reduction=reduction, zero_infinity=True)


def decode(logp):
    return [ctc_greedy(s) for s in logp.argmax(-1).T.cpu().tolist()]


def save_crnn(m, path):
    from safetensors.torch import save_file
    import json
    sd = {k: v.detach().half().contiguous() if v.is_floating_point() else v.contiguous() for k, v in m.state_dict().items()}
    save_file(sd, path, metadata={"cfg": json.dumps(m.cfg)})


def load_crnn(path, device="cpu"):
    from safetensors import safe_open
    from safetensors.torch import load_file
    import json
    with safe_open(path, "pt") as f:
        cfg = json.loads(f.metadata()["cfg"])
    m = CRNN(tuple(cfg["w"]), cfg["seq_layers"], cfg.get("in_ch", 1))
    m.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in load_file(path).items()})
    return m.to(device).eval().requires_grad_(False)


def pil_batch(imgs):
    """list of PIL RGB images -> [B,3,128,128] in [-1,1]."""
    a = np.stack([np.asarray(im.convert("RGB"), np.float32) for im in imgs]) / 127.5 - 1
    return torch.from_numpy(a).permute(0, 3, 1, 2).contiguous()


@torch.no_grad()
def recognize(model, x, layouts, boxes=None, bs=64):
    """x: [B,3,128,128] in [-1,1] tensor or list of PIL; layouts: list of 'left'/'right'/'below';
    boxes: optional [B,4] text bboxes (default: detect_boxes). -> list of strings."""
    if isinstance(x, (list, tuple)):
        x = pil_batch(x)
    dev = next(model.parameters()).device
    out = []
    for b in range(0, len(x), bs):
        xb = x[b:b + bs].to(dev).float()
        lid = torch.tensor([LAYOUT_ID[l] for l in layouts[b:b + bs]], device=dev)
        bx = detect_boxes(xb, lid) if boxes is None else torch.as_tensor(boxes[b:b + bs]).to(dev)
        out += decode(model(crop_text(xb, bx)))
    return out


In [ ]:
%%writefile /content/grinpress_v3/train_crnn.py
"""Phase 2: train the tiny conv-CTC recognizer on procedural crops from generate_data_v2 (box CPU).

  python train_crnn.py --n_train 20000 --epochs 15            # -> artifacts/crnn.safetensors, assets/crnn_train.json
  python train_crnn.py --eval_only --ckpt artifacts/crnn.safetensors

Data: render_v3 (v3 curriculum texts + 20% uniform random CHARSET strings, uniform layouts/papers/expressions,
50% ink-colour variety), stored as 128x128 grey uint8 + layout + text + GT text bbox + detected text bbox.
Training crops use the GT box or the detected box (50/50) with +-1.5 px jitter; evaluation uses detect_boxes only
(exactly what eval_ocr.py does on generated images). The 48 eval strings are held out of training.
Gate (spec): >= 98% exact match on the 48 procedural reference renders (seeds 1000+i).
"""
import argparse, json, math, os, random, time
from multiprocessing import Pool
import numpy as np, torch, torch.nn.functional as F
from crnn import CRNN, crop_text, detect_boxes, ctc_loss, decode, save_crnn, load_crnn, LAYOUT_ID
from charset import CHARSET

ROOT = os.path.dirname(os.path.abspath(__file__))
_FONTS = None


def fonts():
    global _FONTS
    if _FONTS is None:
        from generate_data import ensure_fonts
        _FONTS = sorted(ensure_fonts())
    return _FONTS


def random_charset_text(rng):
    n = rng.randint(1, 14)
    while True:
        t = "".join(rng.choice(CHARSET) for _ in range(n))
        t = " ".join(t.split())
        if t:
            return t


AMBIG = "cosuvwxzpk"          # glyphs whose lower/upper case differ (almost) only in size
CONFUSE = "ectdrnmhHIl10OoaGgqbB"  # observed / classic confusions (e/c, d/t, I/H, r/e, o/O)


def hard_text(rng):
    """v3 run-3 hard classes (from the run-2b ref48 failure analysis): case-ambiguous x-height strings, confusable
    glyphs, and long 10-14 char strings with spaces/punctuation at tiny font sizes."""
    k = rng.random()
    if k < 0.35:      # case-ambiguous: all lower / all upper / one-cap of size-only-case letters (+ a few others)
        n = rng.randint(2, 8); pool = AMBIG + "aeimnr"
        t = "".join(rng.choice(pool if rng.random() < 0.8 else AMBIG) for _ in range(n))
        r = rng.random(); t = t.upper() if r < 0.4 else (t if r < 0.8 else "".join(c.upper() if rng.random() < 0.4 else c for c in t))
    elif k < 0.6:     # confusables
        n = rng.randint(3, 12); t = "".join(rng.choice(CONFUSE if rng.random() < 0.7 else CHARSET) for _ in range(n))
    else:             # long 10-14, word-ish with spaces, mixed case + punctuation
        n = rng.randint(10, 14); t = ""
        while len(t) < n:
            w = "".join(rng.choice(CHARSET[:52] if rng.random() < 0.85 else CHARSET) for _ in range(rng.randint(2, 7)))
            r = rng.random(); w = w.upper() if r < 0.35 else (w.lower() if r < 0.7 else w)
            t = (t + " " + w) if t else w
        t = t[:n]
    t = " ".join(t.split())
    return t or "ok?"


HARD_FRAC = 0.0


def _render(job):
    i, seed = job[:2]
    hard_frac = job[2] if len(job) > 2 else 0.0
    from generate_data_v2 import render_v3, sample_spec_v3
    from eval_grid import HELD_OUT
    rng = random.Random(seed * 1_000_003 + i)
    spec = sample_spec_v3(rng)
    if rng.random() < 0.2:
        t = random_charset_text(rng)
        spec["text"] = t if t not in HELD_OUT else spec["text"]
    if hard_frac and rng.random() < hard_frac:
        t = hard_text(rng)
        spec["text"] = t if t not in HELD_OUT else spec["text"]
    img, meta, aux = render_v3(128, rng, fonts(), spec=spec, ink_variety=True)
    g = np.asarray(img.convert("L"))
    lid = LAYOUT_ID[meta["layout"]]
    det = detect_boxes(torch.from_numpy(g).float().div(127.5).sub(1)[None, None], torch.tensor([lid]))[0].tolist()
    return g, lid, meta["text"], list(aux["text_bbox"]), det


def make_set(n, seed, workers, cache, hard_frac=0.0):
    """-> (imgs uint8 [n,128,128], layout [n], texts, gt_boxes [n,4], det_boxes [n,4])"""
    if cache and os.path.exists(cache):
        d = np.load(cache, allow_pickle=True)
        if len(d["texts"]) >= n:
            print(f"[data] cached {cache}", flush=True)
            return d["imgs"][:n], d["lay"][:n], list(d["texts"][:n]), d["gt"][:n], d["det"][:n]
    t0 = time.time()
    torch.set_num_threads(1)
    with Pool(workers) as p:
        out = p.map(_render, [(i, seed, hard_frac) for i in range(n)], chunksize=64)
    torch.set_num_threads(os.cpu_count())
    imgs = np.stack([o[0] for o in out]); lay = np.array([o[1] for o in out], np.int64); texts = [o[2] for o in out]
    gt = np.array([o[3] for o in out], np.float32); det = np.array([o[4] for o in out], np.float32)
    print(f"[data] rendered {n} in {time.time()-t0:.0f}s", flush=True)
    if cache:
        os.makedirs(os.path.dirname(cache), exist_ok=True)
        np.savez(cache, imgs=imgs, lay=lay, texts=np.array(texts, dtype=object), gt=gt, det=det)
    return imgs, lay, texts, gt, det


def reference_set():
    """The 48 procedural reference renders of the v2/v3 eval grid (byte-identical to v2's reference_renders)."""
    from generate_data_v2 import render_v2
    from eval_grid import grid48
    imgs, lay, texts, gt = [], [], [], []
    for sp, cap, seed in grid48():
        im, _, aux = render_v2(128, random.Random(seed), fonts(), aux=True, **{k: sp[k] for k in ("color", "expr", "text", "layout", "paper")})
        imgs.append(np.asarray(im.convert("L"))); lay.append(LAYOUT_ID[sp["layout"]]); texts.append(sp["text"]); gt.append(aux["text_bbox"])
    imgs, lay = np.stack(imgs), np.array(lay)
    det = detect_boxes(to_tensor(imgs), torch.from_numpy(lay)).numpy()
    return imgs, lay, texts, np.array(gt, np.float32), det


def to_tensor(u8):
    return torch.from_numpy(u8).float().div_(127.5).sub_(1).unsqueeze(1)


def augment(crop, rng_t):
    """augment the grey channel only; the scale channel (if any) is passed through untouched."""
    if crop.shape[1] > 1:
        return torch.cat([augment(crop[:, :1].contiguous(), rng_t), crop[:, 1:]], 1)
    B, d = crop.shape[0], crop.device
    # blur (sigma ~U(0.3,1.2) on half the batch)
    m = torch.rand(B, device=d) < 0.5
    if m.any():
        s = float(torch.empty(1).uniform_(0.3, 1.2)); k = 5
        ax = torch.arange(k, device=d) - k // 2; g = torch.exp(-ax.float() ** 2 / (2 * s * s)); g = g / g.sum()
        ker = (g[:, None] * g[None]).view(1, 1, k, k)
        crop[m] = F.conv2d(F.pad(crop[m], (2, 2, 2, 2), mode="replicate"), ker)
    # down-up resample (softness like a few-step sample)
    m = torch.rand(B, device=d) < 0.3
    if m.any():
        f = float(torch.empty(1).uniform_(0.45, 0.8))
        small = F.interpolate(crop[m], scale_factor=f, mode="bilinear", align_corners=False, antialias=True)
        crop[m] = F.interpolate(small, size=crop.shape[-2:], mode="bilinear", align_corners=False)
    # contrast / brightness / gamma-ish fade
    c = torch.empty(B, 1, 1, 1, device=d).uniform_(0.55, 1.15); b = torch.empty(B, 1, 1, 1, device=d).uniform_(-0.15, 0.15)
    crop = crop * c + b
    # noise
    crop = crop + torch.randn_like(crop) * torch.empty(B, 1, 1, 1, device=d).uniform_(0, 0.08) * (torch.rand(B, 1, 1, 1, device=d) < 0.6)
    return crop


def evaluate(model, imgs, lay, texts, gt, det, bs=256, use="det"):
    model.eval(); preds = []
    boxes = det if use == "det" else gt
    dev = next(model.parameters()).device
    with torch.no_grad():
        for b in range(0, len(texts), bs):
            x = to_tensor(imgs[b:b + bs]).to(dev)
            preds += decode(model(crop_text(x, torch.from_numpy(boxes[b:b + bs]))))
    from eval_ocr import score_text
    rows = [dict(gt=t, pred=p, **score_text(p, t)) for p, t in zip(preds, texts)]
    n = len(rows)
    return dict(n=n, exact=sum(r["exact"] for r in rows) / n, exact_ci=sum(r["exact_ci"] for r in rows) / n,
                char_acc=float(np.mean([r["char_acc"] for r in rows]))), rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n_train", type=int, default=20000)
    ap.add_argument("--n_val", type=int, default=2000)
    ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--bs", type=int, default=128)
    ap.add_argument("--lr", type=float, default=2e-3)
    ap.add_argument("--width", type=int, nargs=5, default=[24, 48, 96, 128, 160])
    ap.add_argument("--seq_layers", type=int, default=3)
    ap.add_argument("--in_ch", type=int, default=1, help="2 = grey + absolute-scale channel (fixes o/O, w/W case confusions)")
    ap.add_argument("--hard_frac", type=float, default=0.0, help="fraction of training texts from hard_text() (failure classes)")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--tag", default="", help="suffix for cache / ckpt names (keeps runs with different data apart)")
    ap.add_argument("--seed", type=int, default=11)
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) - 1))
    ap.add_argument("--threads", type=int, default=os.cpu_count())
    ap.add_argument("--cache_dir", default=os.path.join(ROOT, "cache"))
    ap.add_argument("--out", default=os.path.join(ROOT, "artifacts", "crnn.safetensors"))
    ap.add_argument("--report", default=os.path.join(ROOT, "assets", "crnn_train.json"))
    ap.add_argument("--init", default="", help="continue from an existing crnn.safetensors")
    ap.add_argument("--eval_only", action="store_true")
    ap.add_argument("--ckpt", default="")
    ap.add_argument("--resume_ckpt", default="", help="per-epoch training checkpoint (default cache/crnn_ckpt_*.pt)")
    a = ap.parse_args()
    torch.set_num_threads(a.threads); torch.manual_seed(a.seed)
    ref = reference_set()
    if a.eval_only:
        m = load_crnn(a.ckpt or a.out)
        s, rows = evaluate(m, *ref); print(json.dumps(s, indent=1))
        for r in rows:
            if not r["exact"]: print("  miss:", r["gt"], "->", r["pred"])
        return
    hf = f"_h{int(a.hard_frac*100)}" if a.hard_frac else ""
    tr = make_set(a.n_train, a.seed, a.workers, os.path.join(a.cache_dir, f"crnn_train_s{a.seed}_n{a.n_train}{hf}.npz"), a.hard_frac)
    va = make_set(a.n_val, a.seed + 1000, a.workers, os.path.join(a.cache_dir, f"crnn_val_s{a.seed+1000}_n{a.n_val}.npz"))
    from eval_grid import HELD_OUT
    assert not (set(tr[2]) & HELD_OUT), "eval strings leaked into CRNN training data"
    m = load_crnn(a.init).train().requires_grad_(True) if a.init else CRNN(tuple(a.width), a.seq_layers, a.in_ch)
    dev = torch.device(a.device); m = m.to(dev)
    print(f"[crnn] params {sum(p.numel() for p in m.parameters())/1e6:.2f}M, train {len(tr[2])}, val {len(va[2])}")
    opt = torch.optim.AdamW(m.parameters(), lr=a.lr, weight_decay=1e-4)
    steps_per_ep = len(tr[2]) // a.bs; total = steps_per_ep * a.epochs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=total, pct_start=0.1)
    imgs, lay, texts, gtb, detb = tr; t0 = time.time(); hist = []; step = 0; ep0 = 0
    # per-epoch checkpoint + resume (an interrupted run restarts from the last finished epoch)
    ck = a.resume_ckpt or os.path.join(a.cache_dir, f"crnn_ckpt_s{a.seed}_n{a.n_train}_e{a.epochs}{hf}{a.tag}.pt")
    if os.path.exists(ck):
        st = torch.load(ck, map_location=dev, weights_only=False)
        m.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"])
        hist, step, ep0 = st["hist"], st["step"], st["epoch"]; t0 -= st.get("elapsed", 0)
        torch.set_rng_state(st["rng"].cpu()); print(f"[crnn] resumed {ck} at epoch {ep0}", flush=True)
    for ep in range(ep0, a.epochs):
        m.train(); perm = np.random.default_rng(a.seed + ep).permutation(len(texts)); la = 0
        for k in range(steps_per_ep):
            idx = perm[k * a.bs:(k + 1) * a.bs]
            x = to_tensor(imgs[idx]).to(dev, non_blocking=True)
            pick = torch.rand(len(idx)) < 0.5
            boxes = torch.where(pick[:, None], torch.from_numpy(gtb[idx]), torch.from_numpy(detb[idx])).to(dev)
            jit = torch.empty(len(idx), 4).uniform_(-1.5, 1.5).to(dev)
            crop = augment(crop_text(x, boxes, jit), None)
            loss = ctc_loss(m(crop), [texts[i] for i in idx])
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0); opt.step(); sched.step(); step += 1
            la = 0.98 * la + 0.02 * loss.item() if k else loss.item()
        sv, _ = evaluate(m, *va); sr, _ = evaluate(m, *ref); srg, _ = evaluate(m, *ref, use="gt")
        hist.append(dict(epoch=ep + 1, loss=round(la, 4), val=sv, ref48=sr, ref48_gtbox=srg, elapsed_s=int(time.time() - t0)))
        print(f"[crnn] ep {ep+1}/{a.epochs} loss {la:.4f} | val exact {sv['exact']:.4f} ci {sv['exact_ci']:.4f} | "
              f"ref48 exact {sr['exact']:.4f} ci {sr['exact_ci']:.4f} gtbox {srg['exact']:.4f} | {time.time()-t0:.0f}s", flush=True)
        torch.save(dict(model=m.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), hist=hist, step=step,
                        epoch=ep + 1, elapsed=time.time() - t0, rng=torch.get_rng_state()), ck + ".tmp")
        os.replace(ck + ".tmp", ck)
    m = m.cpu()
    os.makedirs(os.path.dirname(a.out), exist_ok=True); save_crnn(m, a.out)
    m2 = load_crnn(a.out)  # evaluate the fp16-stored weights actually shipped
    sv, _ = evaluate(m2, *va); sr, rows = evaluate(m2, *ref); srg, _ = evaluate(m2, *ref, use="gt")
    rep = dict(args=vars(a), params=sum(p.numel() for p in m.parameters()), history=hist, final_val=sv, final_ref48=sr,
               final_ref48_with_gt_boxes=srg, crop="text box (detect_boxes at eval), 40x200",
               ref48_misses=[r for r in rows if not r["exact"]], gate_ref48_exact_ge_0_98=sr["exact"] >= 0.98,
               train_seconds=int(time.time() - t0))
    json.dump(rep, open(a.report, "w"), indent=1)
    print(json.dumps({k: rep[k] for k in ("final_val", "final_ref48", "gate_ref48_exact_ge_0_98", "train_seconds")}, indent=1))
    for r in rep["ref48_misses"]: print("  miss:", r["gt"], "->", r["pred"])


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/grinpress_v3/ocr_tess.py
"""Tesseract OCR for grinpress renders: the v2 baseline and the improved v3 preprocessing.

v2 baseline (grinpress-v2/eval/eval_samples.py): fixed loose crop -> grey -> 4x Lanczos -> tesseract --psm 7.
v3 improved:
  1. crop the per-layout text band (same boxes as the CRNN, + small margin), upscale 4x (cubic);
  2. estimate the paper colour (median of the brightest 60% of pixels) and score every pixel by its ink-vs-paper
     colour distance (luminance drop + 0.5 x chroma distance), so coloured halftone dots / faint bleed-through
     text score low and near-black ink scores high;
  3. binarise with Otsu on that distance map (Sauvola as an alternative, --binarize sauvola);
  4. clean up: drop specks, drop components touching the face-side border (face outline), tighten to the ink bbox,
     rescale so the text is ~56 px tall, pad with white;
  5. tesseract LSTM with a CHARSET whitelist, --psm 7 and --psm 8; keep the read with the higher mean confidence.
"""
import numpy as np, cv2, pytesseract
from PIL import Image
from charset import CHARSET
from crnn import BANDS, LAYOUT_ID

WHITELIST = CHARSET.replace(" ", "")


def tess_v2(img, layout):
    W = img.width
    box = {"left": (0.44, 0.25, 1.0, 0.75), "right": (0.0, 0.25, 0.56, 0.75), "below": (0.0, 0.58, 1.0, 0.95)}[layout]
    c = img.crop(tuple(int(v * W) for v in box)).convert("L")
    c = c.resize((c.width * 4, c.height * 4), Image.LANCZOS)
    return pytesseract.image_to_string(c, config="--psm 7").strip()


def _sauvola(g, win=31, k=0.2, R=128.0):
    g = g.astype(np.float64)
    m = cv2.boxFilter(g, -1, (win, win)); m2 = cv2.boxFilter(g * g, -1, (win, win))
    s = np.sqrt(np.maximum(m2 - m * m, 0))
    return g > m * (1 + k * (s / R - 1))


def binarize(img, layout, method="otsu", margin=2, up=4):
    a = np.asarray(img.convert("RGB"), np.float32)
    x0, y0, x1, y1 = [int(v) for v in BANDS[LAYOUT_ID[layout]].tolist()]
    x0, y0, x1, y1 = max(0, x0 - margin), max(0, y0 - margin), min(a.shape[1], x1 + margin), min(a.shape[0], y1 + margin)
    c = cv2.resize(a[y0:y1, x0:x1], None, fx=up, fy=up, interpolation=cv2.INTER_CUBIC)
    lum = c @ np.array([0.299, 0.587, 0.114], np.float32)
    bright = lum >= np.percentile(lum, 40)
    paper = np.median(c[bright], 0); plum = float(paper @ np.array([0.299, 0.587, 0.114], np.float32))
    chroma = np.linalg.norm((c - lum[..., None]) - (paper - plum), axis=-1)
    d = np.clip(plum - lum, 0, None) + 0.5 * chroma           # ink-vs-paper colour distance
    d8 = np.clip(d / max(d.max(), 1e-3) * 255, 0, 255).astype(np.uint8)
    if method == "sauvola":
        ink = ~_sauvola(255 - d8) & (d8 > 40)
    else:
        _, b = cv2.threshold(d8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU); ink = b > 0
    ink = ink.astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink, 8)
    keep = np.zeros(n, bool)
    H, W = ink.shape
    for i in range(1, n):
        x, y, w, h, area = st[i]
        if area < 6 * up * up // 4: continue                      # specks
        if layout == "left" and x == 0 and w < W * 0.25: continue  # face outline sliver at the left edge
        if layout == "right" and x + w >= W and w < W * 0.25: continue
        if layout == "below" and y == 0 and h < H * 0.5: continue
        keep[i] = True
    ink = keep[lab]
    if not ink.any():
        return None
    ys, xs = np.nonzero(ink)
    crop = ink[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    s = 56.0 / max(crop.shape[0], 1)
    crop = cv2.resize(crop.astype(np.uint8) * 255, (max(1, int(crop.shape[1] * s)), 56), interpolation=cv2.INTER_AREA)
    out = 255 - cv2.copyMakeBorder(crop, 24, 24, 24, 24, cv2.BORDER_CONSTANT, value=0)
    return Image.fromarray(out)


def _read(b, psm):
    cfg = f"--psm {psm} -c tessedit_char_whitelist={WHITELIST} -c preserve_interword_spaces=1"
    d = pytesseract.image_to_data(b, config=cfg, output_type=pytesseract.Output.DICT)
    words = [(w, float(c)) for w, c in zip(d["text"], d["conf"]) if str(w).strip() and float(c) >= 0]
    if not words:
        return "", -1.0
    return " ".join(w for w, _ in words), float(np.mean([c for _, c in words]))


def tess_v3(img, layout, method="otsu"):
    b = binarize(img, layout, method)
    if b is None:
        return ""
    best = max((_read(b, p) for p in (7, 8)), key=lambda r: r[1])
    return best[0].strip()


In [ ]:
%%writefile /content/grinpress_v3/eval_ocr.py
"""grinpress v3 OCR evaluation.

  python eval_ocr.py refs                       # Phase 2: tesseract v2 / tesseract v3 / CRNN on the 48 reference renders
  python eval_ocr.py models --unet onnx/unet_distilled_int8.onnx --text onnx/text_encoder_int8.onnx [--teacher ckpt]
        # Phase 5: 48-grid at 2 and 4 steps (+14-grid teacher vs student) -> assets/eval_ocr_v3.json, eval_compare_v3.json,
        #          samples_4step_v3.png, samples_2step_v3.png, teacher_vs_student_v3.png

Scores (per string): exact = case-sensitive exact match after whitespace normalisation; exact_ci = v2's metric
(upper-cased, stray face-edge glyph stripped); char_acc = 1 - Levenshtein/len on the exact_ci-normalised strings
(v2 definition), char_acc_cs = same on case-sensitive strings. Primary v3 metric: CRNN exact (case-sensitive).
"""
import argparse, json, os, random, re, sys, time
import numpy as np, torch
from PIL import Image, ImageDraw, ImageFont

ROOT = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, ROOT)


def lev(a, b):
    d = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        p, d[0] = d[0], i
        for j, cb in enumerate(b, 1):
            p, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, p + (ca != cb))
    return d[len(b)]


def norm_ci(s):
    s = re.sub(r"\s+", " ", s.strip().upper())
    return re.sub(r"^[)|}\].,'\u2018\u201c-] ", "", s)


def norm_cs(s):
    return re.sub(r"\s+", " ", s.strip())


def score_text(pred, gt):
    p, g = norm_ci(pred), norm_ci(gt)
    pc, gc = norm_cs(pred), norm_cs(gt)
    return dict(exact=pc == gc, exact_ci=p == g, char_acc=max(0.0, 1 - lev(p, g) / max(1, len(g))),
                char_acc_cs=max(0.0, 1 - lev(pc, gc) / max(1, len(gc))))


def summarize(rows, key):
    P = [r[key] for r in rows]
    return dict(n=len(P), exact=sum(p["exact"] for p in P) / len(P), exact_ci=sum(p["exact_ci"] for p in P) / len(P),
                char_acc=float(np.mean([p["char_acc"] for p in P])), char_acc_cs=float(np.mean([p["char_acc_cs"] for p in P])))


def fonts():
    from generate_data import ensure_fonts
    return sorted(ensure_fonts())


def reference_images(grid):
    from generate_data_v2 import render_v2
    F = fonts()
    return [render_v2(128, random.Random(seed), F, **{k: sp[k] for k in ("color", "expr", "text", "layout", "paper")})[0]
            for sp, _, seed in grid]


def ocr_all(imgs, layouts, crnn=None, tess=("v3",)):
    from crnn import recognize
    from ocr_tess import tess_v2, tess_v3
    out = {}
    if crnn is not None:
        out["crnn"] = recognize(crnn, imgs, layouts)
    if "v2" in tess:
        out["tess_v2"] = [tess_v2(im, l) for im, l in zip(imgs, layouts)]
    if "v3" in tess:
        out["tess_v3"] = [tess_v3(im, l) for im, l in zip(imgs, layouts)]
    if "v3s" in tess:
        out["tess_v3_sauvola"] = [tess_v3(im, l, "sauvola") for im, l in zip(imgs, layouts)]
    return out


FONT = ImageFont.load_default(size=13)


def grid_png(imgs, labels, cols, path, cell=192, title=None):
    rows = (len(imgs) + cols - 1) // cols; lh = 34; th = 30 if title else 0
    G = Image.new("RGB", (cols * cell, th + rows * (cell + lh)), "white"); d = ImageDraw.Draw(G)
    if title: d.text((6, 6), title, fill="black", font=ImageFont.load_default(size=16))
    for k, (im, lab) in enumerate(zip(imgs, labels)):
        r, c = divmod(k, cols); x0, y0 = c * cell, th + r * (cell + lh)
        G.paste(im.resize((cell, cell), Image.NEAREST), (x0, y0))
        for li, line in enumerate(lab.split("\n")[:2]):
            d.text((x0 + 3, y0 + cell + 2 + li * 15), line, fill="black", font=FONT)
    G.save(path); return path


# ------------------------------------------------------------------ Phase 2: references
def cmd_refs(a):
    from eval_grid import grid48
    from crnn import load_crnn
    g = grid48(); imgs = reference_images(g); lays = [sp["layout"] for sp, _, _ in g]; texts = [sp["text"] for sp, _, _ in g]
    crnn = load_crnn(a.crnn) if a.crnn and os.path.exists(a.crnn) else None
    t0 = time.time()
    preds = ocr_all(imgs, lays, crnn, tess=("v2", "v3", "v3s"))
    res = {"what": "OCR on the 48 procedural reference renders (generate_data_v2.render_v2, seeds 1000+i, byte-identical to v2)",
           "crnn": a.crnn if crnn is not None else None, "per_sample": [], "summary": {}}
    for i, (sp, cap, seed) in enumerate(g):
        row = dict(text=sp["text"], layout=sp["layout"], paper=sp["paper"], expr=sp["expr"], seed=seed)
        for k, v in preds.items(): row[k] = dict(ocr=v[i], **score_text(v[i], sp["text"]))
        res["per_sample"].append(row)
    for k in preds: res["summary"][k] = summarize(res["per_sample"], k)
    res["seconds"] = round(time.time() - t0, 1)
    os.makedirs(os.path.dirname(a.out), exist_ok=True)
    json.dump(res, open(a.out, "w"), indent=1)
    print(json.dumps(res["summary"], indent=1))
    for k in preds:
        miss = [(r["text"], r[k]["ocr"]) for r in res["per_sample"] if not r[k]["exact"]]
        print(k, "misses (case-sensitive):", miss[:60])


# ------------------------------------------------------------------ Phase 5: models
def ort_sess(p, threads=os.cpu_count()):
    import onnxruntime as ort
    so = ort.SessionOptions(); so.intra_op_num_threads = threads; so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(p, so, providers=["CPUExecutionProvider"])


def noise(seed):
    return torch.randn(1, 3, 128, 128, generator=torch.Generator().manual_seed(seed)).numpy()


def sched(n):
    return [1.0 - i / n for i in range(n + 1)]


def student_onnx(te, un, caps, seeds, steps, bs=8):
    from conditioning import tokenize
    out = []
    for b in range(0, len(caps), bs):
        c, sd = caps[b:b + bs], seeds[b:b + bs]
        ids, mk = tokenize(c)
        ctx, cm = te.run(None, {"input_ids": ids.numpy(), "attention_mask": mk.long().numpy()})
        x = np.concatenate([noise(s) for s in sd]); ts = sched(steps)
        for i in range(steps):
            v = un.run(None, {"x_t": x, "t": np.full((len(c),), ts[i], np.float32), "context": ctx, "context_mask": cm})[0]
            x = x - (ts[i] - ts[i + 1]) * v
        out.append(np.clip(x, -1, 1))
    return np.concatenate(out)


@torch.no_grad()
def teacher_images(teacher_path, caps, seeds, steps=50, cfg=5.0, bs=8, device="cpu"):
    from train_teacher_v3 import load_flow, sample_cfg
    T = load_flow(teacher_path, device)
    out = []
    for b in range(0, len(caps), bs):
        z = torch.from_numpy(np.concatenate([noise(s) for s in seeds[b:b + bs]])).to(device)
        out.append(sample_cfg(T, caps[b:b + bs], z, steps, cfg).cpu().numpy())
    return np.concatenate(out)


def to_img(x):
    return Image.fromarray(((x.transpose(1, 2, 0) + 1) * 127.5).round().clip(0, 255).astype(np.uint8))


def cmd_models(a):
    from eval_grid import grid48, grid14
    from crnn import load_crnn
    crnn = load_crnn(a.crnn)
    tess = tuple(t for t in a.tess.split(",") if t)
    os.makedirs(a.assets, exist_ok=True)
    te, un = ort_sess(a.text), ort_sess(a.unet)
    g = grid48(); caps = [c for _, c, _ in g]; seeds = [s for *_, s in g]; lays = [sp["layout"] for sp, _, _ in g]
    imgs, res = {}, {"unet": os.path.basename(a.unet), "text_encoder": os.path.basename(a.text), "n": len(g),
                     "captions_truncated": 0, "primary_metric": "crnn exact (case-sensitive)", "per_sample": [], "summary": {}}
    from conditioning import MAX_LEN
    res["captions_truncated"] = sum(len(c) > MAX_LEN for c in caps)
    t0 = time.time()
    for steps in a.steps:
        imgs[f"s{steps}"] = [to_img(z) for z in student_onnx(te, un, caps, seeds, steps)]
    imgs["ref"] = reference_images(g)
    preds = {k: ocr_all(v, lays, crnn, tess) for k, v in imgs.items()}
    for i, (sp, cap, seed) in enumerate(g):
        row = dict(sp, caption=cap, seed=seed)
        for k in imgs:
            row[k] = {m: dict(ocr=p[i], **score_text(p[i], sp["text"])) for m, p in preds[k].items()}
        res["per_sample"].append(row)
    for k in imgs:
        res["summary"][k] = {m: summarize([{"x": r[k][m]} for r in res["per_sample"]], "x") for m in preds[k]}
    res["seconds"] = round(time.time() - t0, 1)
    json.dump(res, open(os.path.join(a.assets, "eval_ocr_v3.json"), "w"), indent=1)
    for steps in a.steps:
        lab = [f"'{sp['text']}' -> '{r[f's{steps}']['crnn']['ocr']}'\n{sp['expr']}/{sp['layout']}/{sp['paper']}"
               for (sp, _, _), r in zip(g, res["per_sample"])]
        grid_png(imgs[f"s{steps}"], lab, 8, os.path.join(a.assets, f"samples_{steps}step_v3.png"),
                 title=f"grinpress v3 student ({os.path.basename(a.unet)}), {steps} steps, seeds 1000+i (label: text -> CRNN read)")
    print(json.dumps(res["summary"], indent=1))
    # ---- 14-string teacher vs student comparison
    g14 = grid14(); c14 = [c for _, c, _ in g14]; s14 = [s for *_, s in g14]; l14 = [sp["layout"] for sp, _, _ in g14]
    rows = {"ref": reference_images(g14)}
    if a.teacher:
        rows["teacher"] = [to_img(z) for z in teacher_images(a.teacher, c14, s14, a.teacher_steps, a.cfg, device=a.device)]
    for steps in a.steps:
        rows[f"s{steps}"] = [to_img(z) for z in student_onnx(te, un, c14, s14, steps)]
    cmp = {}
    for k, ims in rows.items():
        p = ocr_all(ims, l14, crnn, tess)
        cmp[k] = {m: dict(summarize([{"x": score_text(o, sp["text"])} for o, (sp, _, _) in zip(v, g14)], "x"), ocr=v) for m, v in p.items()}
    json.dump(cmp, open(os.path.join(a.assets, "eval_compare_v3.json"), "w"), indent=1)
    all_imgs, labels = [], []
    for k, ims in rows.items():
        for i, im in enumerate(ims):
            all_imgs.append(im); labels.append(f"'{g14[i][0]['text']}' {k}\n-> '{cmp[k]['crnn']['ocr'][i]}'")
    grid_png(all_imgs, labels, len(g14), os.path.join(a.assets, "teacher_vs_student_v3.png"), cell=160,
             title="rows: " + " | ".join(rows) + "  (same noise per column; label: CRNN read)")
    print(json.dumps({k: {m: {x: y for x, y in v.items() if x != "ocr"} for m, v in d.items()} for k, d in cmp.items()}, indent=1))


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    r = sub.add_parser("refs")
    r.add_argument("--crnn", default=os.path.join(ROOT, "artifacts", "crnn.safetensors"))
    r.add_argument("--out", default=os.path.join(ROOT, "assets", "eval_ocr_refs_phase2.json"))
    m = sub.add_parser("models")
    m.add_argument("--unet", required=True)
    m.add_argument("--text", required=True)
    m.add_argument("--teacher", default="")
    m.add_argument("--teacher_steps", type=int, default=50)
    m.add_argument("--cfg", type=float, default=5.0)
    m.add_argument("--steps", type=int, nargs="+", default=[4, 2])
    m.add_argument("--crnn", default=os.path.join(ROOT, "artifacts", "crnn.safetensors"))
    m.add_argument("--tess", default="v3", help="comma list of tesseract variants to also run (v2,v3,v3s) or ''")
    m.add_argument("--assets", default=os.path.join(ROOT, "assets"))
    m.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    torch.set_num_threads(os.cpu_count())
    {"refs": cmd_refs, "models": cmd_models}[a.cmd](a)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/grinpress_v3/train_teacher_v3.py
"""Phase 3: fine-tune the v1 teacher into grinpress-v3-teacher.

  pregen : render N procedural v3 samples (generate_data_v2.render_v3 curriculum) into local .npy shards
           (images uint8, text bbox, layout, caption, text). Deterministic per (seed, index) -> safe to regenerate;
           finished shards are skipped, so the stage resumes.
  train  : warm-start from skyuu72/grinpress (v1) into the v3 model (max_len 128, zero-init quote/glyph embeddings),
           rectified-flow loss  x_t = (1-t) x0 + t z1,  v = z1 - x0,  L_rf = mean((v_hat - v)^2 * (1 + 4 M))  (M = text bbox mask)
           + 0.1 * CTC(frozen CRNN(GT-text-box crop of x0_hat)) for samples with t < 0.5 (x0_hat = x_t - t v_hat), conditional rows only.
           Validation every --val_every steps: EMA teacher, 50 Euler steps, CFG 5, the exact 48-string eval grid
           (noise seeds 1000+i), scored by the CRNN; early stop once exact > --stop_exact (0.95). Also logs a disjoint
           random 48-string grid so the number isn't only measured on the (selection) grid.
Robustness: local-SSD ckpts + async sharded Drive mirror (<100 MB files), resume from newest, status heartbeat,
micro-batch halving on CUDA OOM (persisted), DONE marker.
"""
import argparse, copy, glob, json, math, os, random, time
import numpy as np, torch, torch.nn.functional as F
from conditioning import tokenize
from model import FlowModel, load_teacher_weights, load_state
from gp_io import (Heartbeat, AsyncMirror, atomic_save, save_json, load_json, list_ckpts, latest_resume_state,
                   is_oom, free_cuda, amp_ctx, write_marker, fake_oom_hook)

RES = 128
LAYOUTS = ["left", "right", "below"]


# ------------------------------------------------------------------ model helpers (also used by eval / distill)
def load_flow(path, device="cpu"):
    m = FlowModel()
    load_teacher_weights(m, path)
    return m.to(device).eval().requires_grad_(False)


def encode_both(model, ids, mask):
    ctx_c, m_c = model.cond(ids, mask)          # call on an eval()-mode model (no random CFG dropout)
    ctx_u, m_u = model.cond(ids, mask, force_uncond=True)
    return ctx_c, m_c, ctx_u, m_u


@torch.no_grad()
def cfg_velocity(model, x, t, conds, w):
    ctx_c, m_c, ctx_u, m_u = conds
    v = model.unet(torch.cat([x, x]), torch.cat([t, t]), torch.cat([ctx_c, ctx_u]), torch.cat([m_c, m_u]))
    v_c, v_u = v.float().chunk(2)
    w = w.view(-1, 1, 1, 1) if torch.is_tensor(w) else w
    return v_u + w * (v_c - v_u)


@torch.no_grad()
def sample_cfg(model, caps, z, steps=50, cfg=5.0, amp=None):
    """Euler t: 1 -> 0 with CFG (same schedule as v1 sample.py / v2 eval)."""
    dev = z.device
    ids, mk = tokenize(caps); ids, mk = ids.to(dev), mk.to(dev)
    with (amp or torch.autocast("cpu", enabled=False)):
        conds = encode_both(model, ids, mk)
        x = z.clone(); ts = torch.linspace(1.0, 0.0, steps + 1, device=dev)
        for i in range(steps):
            tb = ts[i].expand(len(z))
            x = x - (ts[i] - ts[i + 1]) * cfg_velocity(model, x, tb, conds, cfg)
    return x.float().clamp(-1, 1)


# ------------------------------------------------------------------ pregen
_FONTS = None


def _fonts():
    global _FONTS
    if _FONTS is None:
        from generate_data import ensure_fonts
        _FONTS = sorted(ensure_fonts())
    return _FONTS


def _render_job(job):
    i, seed = job
    from generate_data_v2 import render_v3
    rng = random.Random(seed * 1_000_003 + i)
    img, meta, aux = render_v3(RES, rng, _fonts())
    return np.asarray(img, np.uint8), aux["text_bbox"], LAYOUTS.index(meta["layout"]), meta["caption"], meta["text"]


def cmd_pregen(a):
    from multiprocessing import Pool
    os.makedirs(a.data_dir, exist_ok=True)
    _fonts()
    n_sh = math.ceil(a.num_samples / a.shard_size); hb = Heartbeat(a.heartbeat, 60); t0 = time.time()
    with Pool(a.workers) as pool:
        for s in range(n_sh):
            base = os.path.join(a.data_dir, f"shard_{s:04d}")
            if os.path.exists(base + ".json"):
                continue
            lo, hi = s * a.shard_size, min(a.num_samples, (s + 1) * a.shard_size)
            out = pool.map(_render_job, [(i, a.seed) for i in range(lo, hi)], chunksize=32)
            np.save(base + ".tmp.npy", np.stack([o[0] for o in out])); os.replace(base + ".tmp.npy", base + ".npy")
            meta = dict(bbox=[list(o[1]) for o in out], layout=[o[2] for o in out], caption=[o[3] for o in out], text=[o[4] for o in out])
            with open(base + ".json.tmp", "w") as f: json.dump(meta, f)
            os.replace(base + ".json.tmp", base + ".json")       # json last = shard complete
            hb(phase="pregen", shard=s + 1, n_shards=n_sh, elapsed_s=int(time.time() - t0))
            print(f"[pregen] shard {s+1}/{n_sh} ({hi-lo} samples, {time.time()-t0:.0f}s)", flush=True)
    hb(force=True, phase="pregen_done", n_shards=n_sh)


class ShardData:
    """Memory-mapped pregen shards; batches are drawn uniformly (the curriculum lives in the generator)."""
    def __init__(self, data_dir):
        self.imgs, self.meta = [], dict(bbox=[], layout=[], caption=[], text=[])
        for j in sorted(glob.glob(os.path.join(data_dir, "shard_*.json"))):
            self.imgs.append(np.load(j[:-5] + ".npy", mmap_mode="r"))
            m = json.load(open(j))
            for k in self.meta: self.meta[k] += m[k]
        assert self.imgs, f"no shards in {data_dir}"
        self.offsets = np.cumsum([0] + [len(x) for x in self.imgs])
        self.n = int(self.offsets[-1])
        self.bbox = np.array(self.meta["bbox"], np.int64); self.layout = np.array(self.meta["layout"], np.int64)

    def batch(self, idx):
        imgs = []
        for i in idx:
            s = int(np.searchsorted(self.offsets, i, side="right") - 1)
            imgs.append(self.imgs[s][i - self.offsets[s]])
        x0 = torch.from_numpy(np.stack(imgs)).permute(0, 3, 1, 2).float().div_(127.5).sub_(1)
        M = torch.zeros(len(idx), 1, RES, RES)
        for k, i in enumerate(idx):
            x0b, y0b, x1b, y1b = self.bbox[i]; M[k, :, y0b:y1b, x0b:x1b] = 1
        return (x0, M, [self.meta["caption"][i] for i in idx], [self.meta["text"][i] for i in idx], torch.from_numpy(self.layout[idx]),
                torch.from_numpy(self.bbox[idx]).float())


class Prefetch:
    """Background thread assembling batches from the memmap shards so the GPU does not wait on numpy."""
    def __init__(self, ds, bs, seed, depth=4):
        import queue, threading
        self.q = queue.Queue(depth); self.ds, self.bs = ds, bs
        self.rng = np.random.default_rng(seed)
        threading.Thread(target=self._run, daemon=True).start()

    def _run(self):
        while True:
            self.q.put(self.ds.batch(self.rng.integers(0, self.ds.n, self.bs)))

    def __next__(self):
        return self.q.get()


# ------------------------------------------------------------------ validation
def val_grids(n_random=48, seed=4242, limit=0):
    from eval_grid import grid48
    from generate_data_v2 import sample_spec_v3, caption_v2
    g = grid48()
    rng = random.Random(seed); rnd = []
    for i in range(n_random):
        sp = sample_spec_v3(rng)
        rnd.append((sp, caption_v2(sp["color"], sp["expr"], sp["text"], sp["layout"], sp["paper"]), 5000 + i))
    if limit:
        g, rnd = g[:limit], rnd[:limit]
    return {"grid48": g, "random48": rnd}


@torch.no_grad()
def validate(model, crnn, grids, dev, steps, cfg, amp, bs=48):
    from crnn import recognize
    from eval_ocr import score_text
    out = {}
    for name, g in grids.items():
        caps = [c for _, c, _ in g]; seeds = [s for *_, s in g]; texts = [sp["text"] for sp, _, _ in g]
        xs = []
        for b in range(0, len(g), bs):
            z = torch.cat([torch.randn(1, 3, RES, RES, generator=torch.Generator().manual_seed(s)) for s in seeds[b:b + bs]]).to(dev)
            xs.append(sample_cfg(model, caps[b:b + bs], z, steps, cfg, amp))
        x = torch.cat(xs)
        preds = recognize(crnn, x, [sp["layout"] for sp, _, _ in g])
        sc = [score_text(p, t) for p, t in zip(preds, texts)]
        out[name] = dict(exact=sum(s["exact"] for s in sc) / len(sc), exact_ci=sum(s["exact_ci"] for s in sc) / len(sc),
                         char_acc_cs=float(np.mean([s["char_acc_cs"] for s in sc])), reads=preds)
    return out, x


# ------------------------------------------------------------------ train
def logit_normal(n, dev):
    return torch.sigmoid(torch.randn(n, device=dev)).clamp(1e-4, 1.0)


def cmd_train(a):
    from crnn import load_crnn, crop_text, ctc_loss
    torch.manual_seed(a.seed)
    dev = torch.device(a.device)
    os.makedirs(a.checkpoint_dir, exist_ok=True); os.makedirs(a.out_dir, exist_ok=True)
    hb = Heartbeat(a.heartbeat, a.heartbeat_sec); mirror = AsyncMirror(a.checkpoint_mirror, keep=2)
    model = FlowModel(); load_teacher_weights(model, a.init)
    model.cond.p_uncond = 0.0                                   # CFG dropout is done here (we need to know which rows)
    model.to(dev).train()
    ema = copy.deepcopy(model).eval().requires_grad_(False)
    crnn = load_crnn(a.crnn, dev)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, betas=(0.9, 0.99), weight_decay=0.0)
    ac, scaler = amp_ctx(dev, a.precision)
    step, best, hist = 0, -1.0, []
    st, src = (latest_resume_state(a.checkpoint_dir, a.checkpoint_mirror) if a.resume else (None, None))
    if st is not None:
        model.load_state_dict(st["model"]); ema.load_state_dict(st["ema"]); opt.load_state_dict(st["opt"])
        if scaler and st.get("scaler"): scaler.load_state_dict(st["scaler"])
        step, best, hist = st["step"], st.get("best", -1.0), st.get("hist", [])
        print(f"[teacher] resumed {src} at step {step} (best grid48 exact {best:.3f})", flush=True)
        del st
    hist_path = os.path.join(a.out_dir, "teacher_val_history.json")
    stop = any(h.get("early_stop") for h in hist)
    if stop:
        print("[teacher] early-stop already reached in a previous run; finalising", flush=True)
    ds = ShardData(a.data_dir); it = Prefetch(ds, a.batch_size, a.seed + step)
    grids = val_grids(limit=a.val_limit)
    mpath = os.path.join(a.checkpoint_dir, "micro.json")
    micro = min(a.batch_size, a.micro_batch or a.batch_size, load_json(mpath, {}).get("micro_batch", 10 ** 9))
    print(f"[teacher] {ds.n} samples, batch {a.batch_size} (micro {micro}), lr {a.lr}, max_steps {a.max_steps}", flush=True)

    def save_ckpt():
        p = os.path.join(a.checkpoint_dir, f"ckpt_{step:08d}.pt")
        atomic_save({"model": model.state_dict(), "ema": ema.state_dict(), "opt": opt.state_dict(),
                     "scaler": scaler.state_dict() if scaler else None, "step": step, "best": best, "hist": hist,
                     "args": vars(a)}, p)
        for old in list_ckpts(a.checkpoint_dir)[:-2]:
            try: os.remove(old)
            except OSError: pass
        mirror(p)

    def export_ema(name):
        from safetensors.torch import save_file
        sd = {k: v.detach().float().cpu().contiguous() for k, v in ema.state_dict().items()}
        loc = os.path.join(a.checkpoint_dir, name + ".safetensors"); save_file(sd, loc + ".tmp"); os.replace(loc + ".tmp", loc)
        fp16 = os.path.join(a.out_dir, name + "_fp16.safetensors")     # 68 MB: under the 100 MB Drive cap
        def _w():
            save_file({k: v.half() for k, v in sd.items()}, fp16 + ".tmp"); os.replace(fp16 + ".tmp", fp16)
        from gp_io import drive_retry
        try: drive_retry(_w, f"write {os.path.basename(fp16)}")
        except Exception as err: print(f"[teacher] fp16 export failed ({err})", flush=True)
        return loc

    t_start, last_save, la, step0 = time.time(), time.time(), None, step
    while step < a.max_steps and not stop:
        x0, M, caps, texts, lay, boxes = next(it)
        x0, M, lay, boxes = x0.to(dev, non_blocking=True), M.to(dev), lay.to(dev), boxes.to(dev)
        B = x0.shape[0]
        ids, mk = tokenize(caps); ids, mk = ids.to(dev), mk.to(dev)
        z1 = torch.randn_like(x0); t = logit_normal(B, dev); tb = t.view(-1, 1, 1, 1)
        xt = (1 - tb) * x0 + tb * z1; v = z1 - x0
        drop = torch.rand(B, device=dev) < a.p_uncond
        wmap = 1 + a.mask_weight * M
        for g in opt.param_groups: g["lr"] = a.lr * min(1.0, (step + 1) / a.warmup_steps)
        while True:
            try:
                opt.zero_grad(set_to_none=True); ls = {"rf": 0.0, "ctc": 0.0}; fake_oom_hook(micro)
                for i in range(0, B, micro):
                    sl = slice(i, i + micro); frac = min(micro, B - i) / B
                    with ac:
                        ctx, m = model.cond(ids[sl], mk[sl])
                        nx, nm = model.cond.null(ctx.shape[0], ctx.shape[1], ctx.dtype, ctx.device)
                        d = drop[sl]
                        ctx = torch.where(d[:, None, None], nx, ctx); m = torch.where(d[:, None], nm, m)
                        vh = model.unet(xt[sl], t[sl], ctx, m)
                    l_rf = ((vh.float() - v[sl]) ** 2 * wmap[sl]).mean()
                    loss = l_rf
                    sel = (t[sl] < a.ctc_t_max) & ~d
                    if a.ctc_weight > 0 and sel.any():
                        x0h = (xt[sl][sel] - t[sl][sel].view(-1, 1, 1, 1) * vh[sel].float()).clamp(-1, 1)
                        lp = crnn(crop_text(x0h, boxes[sl][sel]))        # crop the GT text box
                        l_ctc = ctc_loss(lp, [texts[j] for j in range(sl.start, min(sl.stop, B)) if sel[j - sl.start]])
                        loss = loss + a.ctc_weight * l_ctc * sel.float().mean()
                        ls["ctc"] += float(l_ctc.detach()) * frac
                    ls["rf"] += float(l_rf.detach()) * frac
                    (scaler.scale(loss * frac) if scaler else loss * frac).backward()
                break
            except Exception as err:
                if not is_oom(err) or micro <= 1: raise
                opt.zero_grad(set_to_none=True); free_cuda(); micro = max(1, micro // 2)
                save_json({"micro_batch": micro, "reason": "oom"}, mpath)
                print(f"[teacher] CUDA OOM -> micro-batch {micro}, retrying step {step}", flush=True)
        if scaler:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        with torch.no_grad():
            dec = min(a.ema_decay, (1 + step) / (10 + step))
            for pe, p in zip(ema.parameters(), model.parameters()): pe.lerp_(p.detach(), 1 - dec)
        step += 1
        la = ls["rf"] if la is None else 0.98 * la + 0.02 * ls["rf"]
        if step % a.log_every == 0 or step == 1:
            print(f"[teacher] step {step}/{a.max_steps} rf {ls['rf']:.4f} (avg {la:.4f}) ctc {ls['ctc']:.3f} "
                  f"{(time.time()-t_start)/max(1, step - step0):.2f}s/it", flush=True)
        hb(phase="teacher_train", step=step, max_steps=a.max_steps, rf=round(la, 5), ctc=round(ls["ctc"], 4), best=best)
        if a.val_every and step % a.val_every == 0:
            tv = time.time()
            r, x = validate(ema, crnn, grids, dev, a.val_steps, a.cfg, ac)
            rec = dict(step=step, grid48=r["grid48"]["exact"], grid48_ci=r["grid48"]["exact_ci"], random48=r["random48"]["exact"],
                       random48_ci=r["random48"]["exact_ci"], reads48=r["grid48"]["reads"], val_s=round(time.time() - tv, 1))
            if rec["grid48"] > best:
                best = rec["grid48"]; export_ema("teacher_v3_best"); rec["new_best"] = True
            if rec["grid48"] > a.stop_exact:
                rec["early_stop"] = True; stop = True
            hist.append(rec)
            try: save_json(hist, hist_path)
            except Exception as err: print(f"[teacher] history write failed ({err})", flush=True)
            print(f"[teacher] VAL step {step}: grid48 exact {rec['grid48']:.3f} (ci {rec['grid48_ci']:.3f}) | random48 exact "
                  f"{rec['random48']:.3f} | best {best:.3f} | {rec['val_s']}s", flush=True)
            hb(force=True, phase="teacher_val", step=step, grid48=rec["grid48"], random48=rec["random48"], best=best)
        if time.time() - last_save > a.save_every_min * 60 or stop:
            save_ckpt(); last_save = time.time()
    if not os.path.exists(os.path.join(a.checkpoint_dir, f"ckpt_{step:08d}.pt")):
        save_ckpt()
    mirror.flush()
    final = export_ema("teacher_v3_last")
    best_local = os.path.join(a.checkpoint_dir, "teacher_v3_best.safetensors")
    chosen = best_local if os.path.exists(best_local) else final
    import shutil
    shutil.copy(chosen, os.path.join(a.checkpoint_dir, "teacher_v3.safetensors"))
    from safetensors.torch import load_file, save_file
    from gp_io import drive_retry
    fp16 = os.path.join(a.out_dir, "teacher_v3_fp16.safetensors")       # the teacher used by Phase 4 (68 MB)
    sd16 = {k: v.half() for k, v in load_file(chosen).items()}
    drive_retry(lambda: (save_file(sd16, fp16 + ".tmp"), os.replace(fp16 + ".tmp", fp16)), "write teacher_v3_fp16")
    write_marker(a.done_marker, step=step, best_grid48_exact=best, teacher=os.path.basename(chosen)) if a.done_marker else None
    hb(force=True, phase="teacher_done", step=step, best=best)
    print(f"[teacher] done at step {step}; best grid48 exact {best:.3f}; teacher -> {chosen}", flush=True)


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    p = sub.add_parser("pregen")
    p.add_argument("--data_dir", default="teacher_data")
    p.add_argument("--num_samples", type=int, default=100000)
    p.add_argument("--shard_size", type=int, default=5000)
    p.add_argument("--seed", type=int, default=3)
    p.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) - 1))
    p.add_argument("--heartbeat", default="")
    t = sub.add_parser("train")
    t.add_argument("--init", default="weights/v1_teacher.safetensors")
    t.add_argument("--crnn", default="artifacts/crnn.safetensors")
    t.add_argument("--data_dir", default="teacher_data")
    t.add_argument("--checkpoint_dir", default="teacher_ckpt")
    t.add_argument("--checkpoint_mirror", default="")
    t.add_argument("--out_dir", default="teacher_out", help="small outputs (fp16 weights, val history); Drive in Colab")
    t.add_argument("--batch_size", type=int, default=128)
    t.add_argument("--micro_batch", type=int, default=0)
    t.add_argument("--lr", type=float, default=1e-4)
    t.add_argument("--warmup_steps", type=int, default=500)
    t.add_argument("--max_steps", type=int, default=40000)
    t.add_argument("--ema_decay", type=float, default=0.9995)
    t.add_argument("--p_uncond", type=float, default=0.1)
    t.add_argument("--mask_weight", type=float, default=4.0)
    t.add_argument("--ctc_weight", type=float, default=0.1)
    t.add_argument("--ctc_t_max", type=float, default=0.5)
    t.add_argument("--val_every", type=int, default=1000)
    t.add_argument("--val_steps", type=int, default=50)
    t.add_argument("--val_limit", type=int, default=0, help="smoke tests: only the first N strings of each val grid")
    t.add_argument("--cfg", type=float, default=5.0)
    t.add_argument("--stop_exact", type=float, default=0.95)
    t.add_argument("--precision", default="auto", choices=["auto", "bf16", "fp16", "fp32"])
    t.add_argument("--save_every_min", type=float, default=10.0)
    t.add_argument("--heartbeat", default="")
    t.add_argument("--heartbeat_sec", type=float, default=60)
    t.add_argument("--done_marker", default="")
    t.add_argument("--log_every", type=int, default=100)
    t.add_argument("--resume", type=int, default=1)
    t.add_argument("--seed", type=int, default=0)
    t.add_argument("--device", default=dev)
    a = ap.parse_args()
    {"pregen": cmd_pregen, "train": cmd_train}[a.cmd](a)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/grinpress_v3/distill_v3.py
"""Phase 4: v3 reflow pairs from the v3 teacher + 2-4 step student distillation.

  gen_pairs : z1 ~ N(0,I) (stored as per-sample seed), x0 = TeacherODE(z1) with 50 Euler steps + CFG 5, captions from
              the v3 curriculum (sample_spec_v3). Shards of --shard_size pairs (1000 -> ~49 MB, under the 100 MB Drive cap),
              each built from atomic chunk files (crash loses at most one chunk), batch halved on CUDA OOM and persisted.
              Per pair we also store the layout and a text bbox measured on the teacher image with crnn.detect_boxes
              (dark-ink bbox inside the layout's text band, the same detector the evaluator uses) -> text mask M + CTC crop.
  train     : student = teacher copy (+ zero-init guidance embedding, w=5 baked in). Per batch
                t ~ 60% {1.0, 0.75, 0.5, 0.25} / 40% logit-normal;  x_t = (1-t) x0 + t z1;  v = z1 - x0;  x0_hat = x_t - t v_hat
                L = mean((v_hat - v)^2 (1 + 4M))                           weighted reflow MSE
                  + l_sobel * mean(|Sobel(x0_hat) - Sobel(x0)| * M) / mean(M)  edge sharpness inside the text box
                  + l_ctc * CTC(frozen CRNN(text-box crop of x0_hat))      (rows with t <= --ctc_t_max)
              on --p_rollout (25%) of steps, also a 2-step rollout z1 -(t=1)-> x_0.5 -(t=0.5)-> x0_2 with gradients through
              both steps: + l_roll * mean((x0_2 - x0)^2 (1 + 4M)) + l_ctc * CTC(CRNN(x0_2)).
  sample    : few-step Euler with the student.
Robustness identical to train_teacher_v3 (local ckpts + async sharded Drive mirror, resume, heartbeat, OOM halving).
"""
import argparse, copy, glob, json, math, os, random, time
import numpy as np, torch, torch.nn.functional as F
from conditioning import tokenize
from model import StudentFlowModel, load_teacher_weights, load_state
from gp_io import (Heartbeat, AsyncMirror, atomic_save, save_json, load_json, list_ckpts, latest_resume_state,
                   is_oom, free_cuda, amp_ctx, write_marker, drive_retry, fake_oom_hook)
from train_teacher_v3 import load_flow, encode_both, cfg_velocity, LAYOUTS, val_grids

RES = 128


def noise_from_seeds(seeds):
    out = torch.empty(len(seeds), 3, RES, RES)
    for i, s in enumerate(seeds):
        out[i] = torch.randn(3, RES, RES, generator=torch.Generator().manual_seed(int(s)))
    return out


def student_schedule(n):
    return [1.0 - i / n for i in range(n + 1)]


class TimeSampler:
    """p_discrete from the few-step grid points, else logit-normal(0,1)."""
    def __init__(self, p_discrete=0.6, points=(1.0, 0.75, 0.5, 0.25)):
        self.p, self.points = p_discrete, torch.tensor(points)

    def __call__(self, n, device):
        t = torch.sigmoid(torch.randn(n))
        d = torch.rand(n) < self.p
        t[d] = self.points[torch.randint(len(self.points), (int(d.sum()),))]
        return t.clamp(1e-3, 1.0).to(device)


@torch.no_grad()
def ink_bbox(x8, layout_ids):
    """Text bbox on teacher outputs (uint8 [n,3,128,128]) with the same detector the evaluator uses."""
    from crnn import detect_boxes
    out = [detect_boxes(x8[b:b + 256].float() / 127.5 - 1, layout_ids[b:b + 256]) for b in range(0, len(x8), 256)]
    return torch.cat(out).round().to(torch.int16)


# ------------------------------------------------------------------ gen_pairs
def shard_specs(seed, s, n):
    from generate_data_v2 import sample_spec_v3, caption_v2
    rng = random.Random(seed * 7919 + s); sp = [sample_spec_v3(rng) for _ in range(n)]
    caps = [caption_v2(p["color"], p["expr"], p["text"], p["layout"], p["paper"]) for p in sp]
    return sp, caps


def cmd_gen_pairs(a):
    dev = torch.device(a.device)
    teacher = load_flow(a.teacher, dev)
    os.makedirs(a.out_dir, exist_ok=True)
    hb = Heartbeat(a.heartbeat, 60); ac, _ = amp_ctx(dev, a.precision)
    n_shards = math.ceil(a.num_pairs / a.shard_size)
    bpath = os.path.join(a.out_dir, "gen_batch.json")
    batch = min(a.batch_size, load_json(bpath, {}).get("batch_size", a.batch_size))
    t0 = time.time()
    for s in range(n_shards):
        path = os.path.join(a.out_dir, f"pairs_{s:04d}.pt")
        if os.path.exists(path):
            continue
        n = min(a.shard_size, a.num_pairs - s * a.shard_size)
        specs, caps = shard_specs(a.seed, s, n)
        seeds = [a.seed * 10_000_000 + s * a.shard_size + i for i in range(n)]
        lay = torch.tensor([LAYOUTS.index(p["layout"]) for p in specs])
        pdir = os.path.join(a.out_dir, f"partial_{s:04d}")
        drive_retry(lambda: os.makedirs(pdir, exist_ok=True), f"mkdir {os.path.basename(pdir)}")
        chunks = {}
        for f in glob.glob(os.path.join(pdir, "chunk_*.pt")):
            b0, b1 = map(int, os.path.basename(f)[6:-3].split("_")); chunks[b0] = (b1, f)
        b = 0
        while b in chunks: b = chunks[b][0]
        if b: print(f"[gen] shard {s}: resuming at {b}/{n}", flush=True)
        while b < n:
            e = min(n, b + batch)
            try:
                fake_oom_hook(e - b)
                z = noise_from_seeds(seeds[b:e]).to(dev)
                ids, mk = tokenize(caps[b:e]); ids, mk = ids.to(dev), mk.to(dev)
                with torch.inference_mode(), ac:
                    conds = encode_both(teacher, ids, mk)
                    x = z; ts = torch.linspace(1.0, 0.0, a.steps + 1, device=dev)
                    for i in range(a.steps):
                        x = x - (ts[i] - ts[i + 1]) * cfg_velocity(teacher, x, ts[i].expand(len(z)), conds, a.cfg)
                    x8 = ((x.float().clamp(-1, 1) + 1) * 127.5).round().byte().cpu()
                del z, x, conds
            except Exception as err:
                if not is_oom(err) or batch <= 1: raise
                free_cuda(); batch = max(1, batch // 2); save_json({"batch_size": batch, "reason": "oom"}, bpath)
                print(f"[gen] CUDA OOM -> batch {batch}, retrying chunk [{b},{e})", flush=True); continue
            cp = os.path.join(pdir, f"chunk_{b:06d}_{e:06d}.pt")
            atomic_save({"x0": x8}, cp); chunks[b] = (e, cp)
            if not os.path.exists(bpath): save_json({"batch_size": batch}, bpath)
            b = e
            hb(phase="gen_pairs", shard=s, n_shards=n_shards, done_in_shard=b, shard_size=n, batch=batch, elapsed_s=int(time.time() - t0))
            print(f"[gen] shard {s}: {b}/{n} (batch {batch}, {time.time()-t0:.0f}s)", flush=True)
        order, k = [], 0
        while k < n: order.append(chunks[k][1]); k = chunks[k][0]
        x0 = torch.cat([drive_retry(lambda f=f: torch.load(f, weights_only=True)["x0"], f"load {os.path.basename(f)}") for f in order])
        assert len(x0) == n
        atomic_save({"x0": x0, "seeds": torch.tensor(seeds), "captions": caps, "texts": [p["text"] for p in specs],
                     "layout": lay, "bbox": ink_bbox(x0, lay), "cfg": a.cfg, "steps": a.steps}, path)
        try:
            for f in glob.glob(os.path.join(pdir, "*")): os.remove(f)
            os.rmdir(pdir)
        except OSError as err:
            print(f"[gen] cleanup of {pdir} failed ({err}); ignoring", flush=True)
        print(f"[gen] wrote {path} ({n} pairs, {time.time()-t0:.0f}s)", flush=True)
    hb(force=True, phase="gen_pairs_done", n_shards=n_shards, batch=batch)


class PairData:
    def __init__(self, pair_dir):
        files = sorted(glob.glob(os.path.join(pair_dir, "pairs_*.pt"))); assert files, f"no pairs in {pair_dir}"
        x0, self.seeds, self.caps, self.texts, lay, bb = [], [], [], [], [], []
        for f in files:
            d = torch.load(f, weights_only=False)
            x0.append(d["x0"]); self.seeds += d["seeds"].tolist(); self.caps += d["captions"]; self.texts += d["texts"]
            lay.append(d["layout"]); bb.append(d["bbox"]); self.cfg = d["cfg"]
        self.x0, self.lay, self.bbox = torch.cat(x0), torch.cat(lay), torch.cat(bb).long()
        self.ids, self.mask = tokenize(self.caps)
        self.n = len(self.seeds)

    def batch(self, idx):
        x0 = self.x0[idx].float() / 127.5 - 1
        z1 = noise_from_seeds([self.seeds[i] for i in idx])
        M = torch.zeros(len(idx), 1, RES, RES)
        for k, i in enumerate(idx):
            x0b, y0b, x1b, y1b = self.bbox[i].tolist()
            M[k, :, max(0, y0b - 2):y1b + 2, max(0, x0b - 2):x1b + 2] = 1      # detected box is tight: dilate 2 px
        return x0, z1, M, self.ids[idx], self.mask[idx], self.bbox[idx].float(), [self.texts[i] for i in idx]


class Prefetch:
    def __init__(self, ds, bs, seed, depth=4):
        import queue, threading
        self.q, self.ds, self.bs, self.rng = queue.Queue(depth), ds, bs, np.random.default_rng(seed)
        threading.Thread(target=self._run, daemon=True).start()

    def _run(self):
        while True:
            self.q.put(self.ds.batch(torch.from_numpy(self.rng.integers(0, self.ds.n, self.bs))))

    def __next__(self):
        return self.q.get()


def sobel(x):
    g = x.mean(1, keepdim=True)
    k = torch.tensor([[1., 0., -1.], [2., 0., -2.], [1., 0., -1.]], device=x.device, dtype=x.dtype)
    gx = F.conv2d(F.pad(g, (1, 1, 1, 1), mode="replicate"), k.view(1, 1, 3, 3))
    gy = F.conv2d(F.pad(g, (1, 1, 1, 1), mode="replicate"), k.t().contiguous().view(1, 1, 3, 3))
    return torch.cat([gx, gy], 1)


@torch.no_grad()
def student_sample(student, caps, z, steps, w=None):
    dev = z.device
    ids, mk = tokenize(caps); ctx, m = student.cond(ids.to(dev), mk.to(dev))
    x = z.clone(); ts = student_schedule(steps)
    for i in range(steps):
        x = x - (ts[i] - ts[i + 1]) * student.forward_ctx(x, torch.full((len(z),), ts[i], device=dev), ctx, m, w=w).float()
    return x.clamp(-1, 1)


@torch.no_grad()
def validate_student(student, crnn, grid, dev, steps_list=(4, 2)):
    from crnn import recognize
    from eval_ocr import score_text
    caps = [c for _, c, _ in grid]; seeds = [s for *_, s in grid]; texts = [sp["text"] for sp, _, _ in grid]
    z = noise_from_seeds(seeds).to(dev); out = {}
    for n in steps_list:
        x = student_sample(student, caps, z, n)
        reads = recognize(crnn, x, [sp["layout"] for sp, _, _ in grid])
        sc = [score_text(p, t) for p, t in zip(reads, texts)]
        out[f"s{n}"] = dict(exact=sum(s["exact"] for s in sc) / len(sc), exact_ci=sum(s["exact_ci"] for s in sc) / len(sc))
    return out


def cmd_train(a):
    from crnn import load_crnn, crop_text, ctc_loss
    torch.manual_seed(a.seed)
    dev = torch.device(a.device)
    os.makedirs(a.checkpoint_dir, exist_ok=True); os.makedirs(a.out_dir, exist_ok=True)
    hb = Heartbeat(a.heartbeat, a.heartbeat_sec); mirror = AsyncMirror(a.checkpoint_mirror, keep=2)
    student = StudentFlowModel(use_w_embed=True, w_default=a.w)
    load_teacher_weights(student, a.teacher)
    student.cond.p_uncond = 0.0
    student.to(dev).train()
    ema = copy.deepcopy(student).eval().requires_grad_(False)
    crnn = load_crnn(a.crnn, dev)
    opt = torch.optim.AdamW(student.parameters(), lr=a.lr, betas=(0.9, 0.99), weight_decay=0.0)
    ac, scaler = amp_ctx(dev, a.precision)
    step, hist = 0, []
    st, src = (latest_resume_state(a.checkpoint_dir, a.checkpoint_mirror) if a.resume else (None, None))
    if st is not None:
        student.load_state_dict(st["model"]); ema.load_state_dict(st["ema"]); opt.load_state_dict(st["opt"])
        if scaler and st.get("scaler"): scaler.load_state_dict(st["scaler"])
        step, hist = st["step"], st.get("hist", []); print(f"[student] resumed {src} at step {step}", flush=True); del st
    ds = PairData(a.pair_dir); it = Prefetch(ds, a.batch_size, a.seed + step)
    tsamp = TimeSampler(a.p_discrete)
    grid = val_grids(limit=a.val_limit)["grid48"]
    mpath = os.path.join(a.checkpoint_dir, "micro.json")
    micro = min(a.batch_size, a.micro_batch or a.batch_size, load_json(mpath, {}).get("micro_batch", 10 ** 9))
    print(f"[student] {ds.n} pairs (cfg {ds.cfg}), batch {a.batch_size} (micro {micro}), max_steps {a.max_steps}", flush=True)
    rng = random.Random(a.seed + step)

    def save_ckpt():
        p = os.path.join(a.checkpoint_dir, f"ckpt_{step:08d}.pt")
        atomic_save({"model": student.state_dict(), "ema": ema.state_dict(), "opt": opt.state_dict(),
                     "scaler": scaler.state_dict() if scaler else None, "step": step, "hist": hist, "args": vars(a)}, p)
        for old in list_ckpts(a.checkpoint_dir)[:-2]:
            try: os.remove(old)
            except OSError: pass
        mirror(p)

    t_start, last_save, la, step0 = time.time(), time.time(), None, step
    while step < a.max_steps:
        x0, z1, M, ids, mk, boxes, texts = next(it)
        x0, z1, M, ids, mk, boxes = (v.to(dev, non_blocking=True) for v in (x0, z1, M, ids, mk, boxes))
        B = x0.shape[0]
        t = tsamp(B, dev); tb = t.view(-1, 1, 1, 1)
        xt = (1 - tb) * x0 + tb * z1; v = z1 - x0; wmap = 1 + a.mask_weight * M
        rollout = rng.random() < a.p_rollout
        wv = torch.full((B,), float(ds.cfg), device=dev)
        for g in opt.param_groups: g["lr"] = a.lr * min(1.0, (step + 1) / a.warmup_steps)
        while True:
            try:
                opt.zero_grad(set_to_none=True); ls = dict(mse=0.0, sobel=0.0, ctc=0.0, roll=0.0); fake_oom_hook(micro)
                for i in range(0, B, micro):
                    sl = slice(i, i + micro); frac = min(micro, B - i) / B
                    with ac:
                        ctx, m = student.cond(ids[sl], mk[sl])
                        vh = student.forward_ctx(xt[sl], t[sl], ctx, m, w=wv[sl]).float()
                    l_mse = ((vh - v[sl]) ** 2 * wmap[sl]).mean()
                    x0h = xt[sl] - t[sl].view(-1, 1, 1, 1) * vh
                    Ms = M[sl]
                    l_sob = ((sobel(x0h) - sobel(x0[sl])).abs() * Ms).mean() / Ms.mean().clamp_min(1e-4)
                    loss = l_mse + a.l_sobel * l_sob
                    sel = t[sl] <= a.ctc_t_max
                    if a.l_ctc > 0 and sel.any():
                        lp = crnn(crop_text(x0h[sel].clamp(-1, 1), boxes[sl][sel]))
                        l_ctc = ctc_loss(lp, [texts[j] for j in range(sl.start, min(sl.stop, B)) if sel[j - sl.start]])
                        loss = loss + a.l_ctc * l_ctc * sel.float().mean(); ls["ctc"] += float(l_ctc.detach()) * frac
                    if rollout:                                    # 2-step consistency 1.0 -> 0.5 -> 0.0
                        one = torch.ones(x0h.shape[0], device=dev)
                        with ac:
                            v1 = student.forward_ctx(z1[sl], one, ctx, m, w=wv[sl]).float()
                            xh = z1[sl] - 0.5 * v1
                            v2 = student.forward_ctx(xh, one * 0.5, ctx, m, w=wv[sl]).float()
                        x02 = xh - 0.5 * v2
                        l_roll = ((x02 - x0[sl]) ** 2 * wmap[sl]).mean()
                        loss = loss + a.l_roll * l_roll; ls["roll"] += float(l_roll.detach()) * frac
                        if a.l_ctc > 0:
                            loss = loss + a.l_ctc * ctc_loss(crnn(crop_text(x02.clamp(-1, 1), boxes[sl])), texts[sl])
                    ls["mse"] += float(l_mse.detach()) * frac; ls["sobel"] += float(l_sob.detach()) * frac
                    (scaler.scale(loss * frac) if scaler else loss * frac).backward()
                break
            except Exception as err:
                if not is_oom(err) or micro <= 1: raise
                opt.zero_grad(set_to_none=True); free_cuda(); micro = max(1, micro // 2)
                save_json({"micro_batch": micro, "reason": "oom"}, mpath)
                print(f"[student] CUDA OOM -> micro-batch {micro}, retrying step {step}", flush=True)
        if scaler:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0); scaler.step(opt); scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0); opt.step()
        with torch.no_grad():
            dec = min(a.ema_decay, (1 + step) / (10 + step))
            for pe, p in zip(ema.parameters(), student.parameters()): pe.lerp_(p.detach(), 1 - dec)
        step += 1
        la = ls["mse"] if la is None else 0.98 * la + 0.02 * ls["mse"]
        if step % a.log_every == 0 or step == 1:
            print(f"[student] step {step}/{a.max_steps} mse {ls['mse']:.4f} (avg {la:.4f}) sobel {ls['sobel']:.3f} "
                  f"ctc {ls['ctc']:.3f} roll {ls['roll']:.4f} {(time.time()-t_start)/max(1, step-step0):.2f}s/it", flush=True)
        hb(phase="student_train", step=step, max_steps=a.max_steps, mse=round(la, 5))
        if a.val_every and step % a.val_every == 0:
            r = validate_student(ema, crnn, grid, dev); r["step"] = step; hist.append(r)
            try: save_json(hist, os.path.join(a.out_dir, "student_val_history.json"))
            except Exception as err: print(f"[student] history write failed ({err})", flush=True)
            print(f"[student] VAL step {step}: 4-step exact {r['s4']['exact']:.3f} | 2-step exact {r['s2']['exact']:.3f}", flush=True)
        if time.time() - last_save > a.save_every_min * 60:
            save_ckpt(); last_save = time.time()
    if not os.path.exists(os.path.join(a.checkpoint_dir, f"ckpt_{step:08d}.pt")):
        save_ckpt()
    mirror.flush()
    from safetensors.torch import save_file
    sd = {k: v.detach().float().cpu().contiguous() for k, v in ema.state_dict().items()}
    loc = os.path.join(a.checkpoint_dir, "student_ema.safetensors"); save_file(sd, loc + ".tmp"); os.replace(loc + ".tmp", loc)
    fp16 = os.path.join(a.out_dir, "student_ema_fp16.safetensors")
    drive_retry(lambda: (save_file({k: v.half() for k, v in sd.items()}, fp16 + ".tmp"), os.replace(fp16 + ".tmp", fp16)), "write student fp16")
    if a.done_marker: write_marker(a.done_marker, step=step)
    hb(force=True, phase="student_done", step=step)
    print(f"[student] done; EMA -> {loc} (+ fp16 {fp16})", flush=True)


def load_student(path, dev="cpu", w_default=5.0):
    s = StudentFlowModel(use_w_embed=True, w_default=w_default)
    sd = load_state(path)
    if not any(k.startswith("w_mlp") for k in sd): s.w_mlp = None
    load_teacher_weights(s, sd)
    return s.to(dev).eval().requires_grad_(False)


def cmd_sample(a):
    from PIL import Image
    s = load_student(a.checkpoint, a.device)
    z = noise_from_seeds([a.seed + i for i in range(a.num_images)]).to(a.device)
    x = student_sample(s, [a.caption] * a.num_images, z, a.steps)
    arr = ((x + 1) * 127.5).round().byte().permute(0, 2, 3, 1).cpu().numpy()
    Image.fromarray(np.concatenate(list(arr), 1)).save(a.out); print("saved", a.out)


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    g = sub.add_parser("gen_pairs")
    g.add_argument("--teacher", required=True)
    g.add_argument("--out_dir", default="pairs_v3")
    g.add_argument("--num_pairs", type=int, default=50000)
    g.add_argument("--shard_size", type=int, default=1000)
    g.add_argument("--steps", type=int, default=50)
    g.add_argument("--cfg", type=float, default=5.0)
    g.add_argument("--batch_size", type=int, default=128)
    g.add_argument("--seed", type=int, default=1)
    g.add_argument("--precision", default="auto", choices=["auto", "bf16", "fp16", "fp32"])
    g.add_argument("--heartbeat", default="")
    g.add_argument("--device", default=dev)
    t = sub.add_parser("train")
    t.add_argument("--teacher", required=True)
    t.add_argument("--crnn", default="artifacts/crnn.safetensors")
    t.add_argument("--pair_dir", default="pairs_v3")
    t.add_argument("--checkpoint_dir", default="student_ckpt")
    t.add_argument("--checkpoint_mirror", default="")
    t.add_argument("--out_dir", default="student_out")
    t.add_argument("--batch_size", type=int, default=128)
    t.add_argument("--micro_batch", type=int, default=0)
    t.add_argument("--lr", type=float, default=5e-5)
    t.add_argument("--warmup_steps", type=int, default=500)
    t.add_argument("--max_steps", type=int, default=20000)
    t.add_argument("--ema_decay", type=float, default=0.999)
    t.add_argument("--p_discrete", type=float, default=0.6)
    t.add_argument("--mask_weight", type=float, default=4.0)
    t.add_argument("--l_sobel", type=float, default=0.25)
    t.add_argument("--l_ctc", type=float, default=0.05)
    t.add_argument("--ctc_t_max", type=float, default=0.75)
    t.add_argument("--p_rollout", type=float, default=0.25)
    t.add_argument("--l_roll", type=float, default=1.0)
    t.add_argument("--w", type=float, default=5.0)
    t.add_argument("--val_every", type=int, default=2000)
    t.add_argument("--val_limit", type=int, default=0)
    t.add_argument("--precision", default="auto", choices=["auto", "bf16", "fp16", "fp32"])
    t.add_argument("--save_every_min", type=float, default=10.0)
    t.add_argument("--heartbeat", default="")
    t.add_argument("--heartbeat_sec", type=float, default=60)
    t.add_argument("--done_marker", default="")
    t.add_argument("--log_every", type=int, default=100)
    t.add_argument("--resume", type=int, default=1)
    t.add_argument("--seed", type=int, default=0)
    t.add_argument("--device", default=dev)
    s = sub.add_parser("sample")
    s.add_argument("--checkpoint", required=True)
    s.add_argument("--caption", default="a red smiley face with 'HELLO WORLD' next to it")
    s.add_argument("--steps", type=int, default=4)
    s.add_argument("--seed", type=int, default=0)
    s.add_argument("--num_images", type=int, default=4)
    s.add_argument("--out", default="student.png")
    s.add_argument("--device", default=dev)
    a = ap.parse_args()
    {"gen_pairs": cmd_gen_pairs, "train": cmd_train, "sample": cmd_sample}[a.cmd](a)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/grinpress_v3/export_v3.py
"""Phase 5: export the v3 student to ONNX (fp32 + dynamic INT8), fp16 safetensors, and validate.

ONNX interface (index.html / bench / eval):
  text_encoder.onnx   : input_ids int64 [B,128] (ASCII code points, 0 = pad, 128 = unk), attention_mask int64 [B,128]
                        -> context float32 [B,128,256], context_mask int64 [B,128]
                        (quoted-span token-type + glyph-index features are computed INSIDE the graph from input_ids)
  unet_distilled.onnx : x_t float32 [B,3,128,128], t float32 [B], context, context_mask -> v_pred float32 [B,3,128,128]
  opset 17, dynamic batch. *_int8.onnx: onnxruntime quantize_dynamic, MatMul/Gemm only (Conv stays fp32).
  student_ema_fp16.safetensors: fp16 copy of the EMA student (~68 MB, under the 100 MB transfer cap).
Files > 95 MB (unet_distilled.onnx fp32 ~130 MB) are additionally split into .partNN + manifest with --split_big.
"""
import argparse, hashlib, json, os, time
import numpy as np, torch, torch.nn as nn
from conditioning import tokenize, MAX_LEN
from distill_v3 import load_student, student_schedule

CAPS = ["a red smiley face with 'HELLO WORLD' next to it, face on the right, on sepia paper",
        "a blue sad face with 'nyaa~' next to it", "a navy uwu face with 'GoodGirl' next to it, text below"]


class TextEncoderExport(nn.Module):
    def __init__(self, s):
        super().__init__(); self.cond = s.cond

    def forward(self, input_ids, attention_mask):
        m = attention_mask > 0
        x = self.cond.embed(input_ids)
        for layer in self.cond.encoder.layers:          # explicit loop: no nested-tensor fast path in the graph
            h = layer.norm1(x)
            x = x + layer.self_attn(h, h, h, key_padding_mask=~m, need_weights=False)[0]
            x = x + layer.linear2(layer.activation(layer.linear1(layer.norm2(x))))
        return self.cond.norm(x), m.to(torch.int64)


class UNetExport(nn.Module):
    def __init__(self, s):
        super().__init__(); self.s = s
        w = s.w_embed(None, 1, "cpu")
        self.register_buffer("w_add", w.detach() if w is not None else torch.zeros(1, s.unet.tmlp[2].out_features))

    def forward(self, x_t, t, context, context_mask):
        return self.s.unet(x_t, t, context, context_mask > 0, temb_add=self.w_add)


def ort_session(path, threads=os.cpu_count()):
    import onnxruntime as ort
    so = ort.SessionOptions(); so.intra_op_num_threads = threads; so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])


def ort_sample(te, un, caption, steps, seed, n=1):
    ids, mk = tokenize([caption] * n)
    ctx, cm = te.run(None, {"input_ids": ids.numpy(), "attention_mask": mk.long().numpy()})
    x = torch.randn(n, 3, 128, 128, generator=torch.Generator().manual_seed(seed)).numpy(); ts = student_schedule(steps)
    for i in range(steps):
        v = un.run(None, {"x_t": x, "t": np.full((n,), ts[i], np.float32), "context": ctx, "context_mask": cm})[0]
        x = x - (ts[i] - ts[i + 1]) * v
    return np.clip(x, -1, 1)


def torch_sample(student, caption, steps, seed, n=1):
    ids, mk = tokenize([caption] * n)
    with torch.inference_mode():
        ctx, m = student.cond(ids, mk); w_add = student.w_embed(None, n, "cpu")
        x = torch.randn(n, 3, 128, 128, generator=torch.Generator().manual_seed(seed)); ts = student_schedule(steps)
        for i in range(steps):
            x = x - (ts[i] - ts[i + 1]) * student.unet(x, torch.full((n,), ts[i]), ctx, m, temb_add=w_add)
    return x.clamp(-1, 1).numpy()


def split_big(path, limit=95 * 1000 * 1000):
    """path -> path.partNN (<= limit bytes) + path.manifest.json (sha256). Reassemble: cat path.part* > path."""
    data = open(path, "rb").read()
    if len(data) <= limit:
        return 0
    n = 0
    for n, i in enumerate(range(0, len(data), limit)):
        with open(f"{path}.part{n:02d}", "wb") as f: f.write(data[i:i + limit])
    json.dump({"n_parts": n + 1, "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest(),
               "reassemble": f"cat {os.path.basename(path)}.part* > {os.path.basename(path)}"}, open(path + ".manifest.json", "w"), indent=1)
    return n + 1


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--student", required=True)
    ap.add_argument("--out_dir", default="onnx")
    ap.add_argument("--report", default="")
    ap.add_argument("--threads", type=int, default=os.cpu_count())
    ap.add_argument("--steps", type=int, default=4)
    ap.add_argument("--opset", type=int, default=17)
    ap.add_argument("--split_big", action="store_true", help="also write <95 MB parts of big files (Drive cap)")
    ap.add_argument("--fp16_out", default="", help="where to write student_ema_fp16.safetensors (default out_dir)")
    a = ap.parse_args()
    torch.set_num_threads(a.threads)
    torch.backends.mha.set_fastpath_enabled(False)   # keep aten::_native_multi_head_attention out of the traced graph
    os.makedirs(a.out_dir, exist_ok=True)
    s = load_student(a.student, "cpu")
    from safetensors.torch import save_file
    fp16 = a.fp16_out or os.path.join(a.out_dir, "student_ema_fp16.safetensors")
    save_file({k: v.half().contiguous() for k, v in s.state_dict().items()}, fp16)
    te, un = TextEncoderExport(s).eval(), UNetExport(s).eval()
    ids, mk = tokenize(CAPS[:2]); mk = mk.long()
    with torch.no_grad(): ctx, cm = te(ids, mk)
    tp, up = os.path.join(a.out_dir, "text_encoder.onnx"), os.path.join(a.out_dir, "unet_distilled.onnx")
    t0 = time.time()
    torch.onnx.export(te, (ids, mk), tp, opset_version=a.opset, dynamo=False,
                      input_names=["input_ids", "attention_mask"], output_names=["context", "context_mask"],
                      dynamic_axes={k: {0: "batch"} for k in ["input_ids", "attention_mask", "context", "context_mask"]})
    torch.onnx.export(un, (torch.randn(2, 3, 128, 128), torch.tensor([1.0, 0.5]), ctx, cm), up, opset_version=a.opset, dynamo=False,
                      input_names=["x_t", "t", "context", "context_mask"], output_names=["v_pred"],
                      dynamic_axes={k: {0: "batch"} for k in ["x_t", "t", "context", "context_mask", "v_pred"]})
    rep = {"opset": a.opset, "max_len": MAX_LEN, "export_s": round(time.time() - t0, 1)}
    # validation: batch 3 (dynamic batch), ORT fp32 vs torch
    ids, mk = tokenize(CAPS); x = torch.randn(3, 3, 128, 128, generator=torch.Generator().manual_seed(0)); t = torch.tensor([1.0, 0.75, 0.25])
    with torch.no_grad():
        ctx_t, cm_t = te(ids, mk.long()); ref_ctx, _ = s.cond(ids, mk); v_t = un(x, t, ctx_t, cm_t).numpy()
    ste, sun = ort_session(tp, a.threads), ort_session(up, a.threads)
    ctx_o, cm_o = ste.run(None, {"input_ids": ids.numpy(), "attention_mask": mk.long().numpy()})
    v_o = sun.run(None, {"x_t": x.numpy(), "t": t.numpy(), "context": ctx_o, "context_mask": cm_o})[0]
    rep["export_wrapper_vs_module_ctx_maxdiff"] = float(((ctx_t - ref_ctx) * mk[..., None]).abs().max())
    rep["ort_fp32_ctx_maxdiff"] = float(np.abs((ctx_o - ctx_t.numpy()) * mk[..., None].numpy()).max())
    rep["ort_fp32_v_maxdiff"] = float(np.abs(v_o - v_t).max())
    rep["context_mask_equal"] = bool((cm_o == cm_t.numpy()).all())
    img_t = torch_sample(s, CAPS[0], a.steps, 3); img_o = ort_sample(ste, sun, CAPS[0], a.steps, 3)
    rep["ort_fp32_image_maxdiff"] = float(np.abs(img_o - img_t).max())
    from onnxruntime.quantization import quantize_dynamic, QuantType
    q = {}
    for p in (tp, up):
        qp = p.replace(".onnx", "_int8.onnx")
        quantize_dynamic(p, qp, weight_type=QuantType.QInt8, op_types_to_quantize=["MatMul", "Gemm"]); q[p] = qp
        for f in (p, qp): rep[f"size_MB_{os.path.basename(f)}"] = round(os.path.getsize(f) / 1e6, 1)
    teq, unq = ort_session(q[tp], a.threads), ort_session(q[up], a.threads)
    ctx_q, cm_q = teq.run(None, {"input_ids": ids.numpy(), "attention_mask": mk.long().numpy()})
    v_q = unq.run(None, {"x_t": x.numpy(), "t": t.numpy(), "context": ctx_q, "context_mask": cm_q})[0]
    rep["ort_int8_v_maxdiff"] = float(np.abs(v_q - v_t).max()); rep["ort_int8_v_meandiff"] = float(np.abs(v_q - v_t).mean())
    img_q = ort_sample(teq, unq, CAPS[0], a.steps, 3)
    rep["ort_int8_image_meandiff_pixels"] = float(np.abs(img_q - img_t).mean() * 127.5)
    rep[f"size_MB_{os.path.basename(fp16)}"] = round(os.path.getsize(fp16) / 1e6, 1)
    if a.split_big:
        rep["split_parts"] = {os.path.basename(p): split_big(p) for p in (tp, up, q[tp], q[up], fp16)}
    from PIL import Image
    to8 = lambda z: ((z[0].transpose(1, 2, 0) + 1) * 127.5).round().clip(0, 255).astype(np.uint8)
    Image.fromarray(np.concatenate([to8(img_t), to8(img_o), to8(img_q)], 1)).save(os.path.join(a.out_dir, "torch_ortfp32_ortint8.png"))
    print(json.dumps(rep, indent=1))
    json.dump(rep, open(a.report or os.path.join(a.out_dir, "export_report.json"), "w"), indent=1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/grinpress_v3/bench_cpu.py
"""CPU latency of the v3 student (batch 1, 128 px, full image = text encoder + all steps), median of --reps after warm-up.
  python bench_cpu.py --onnx_dir onnx [--student student_ema.safetensors] -> assets/bench_cpu_v3.json"""
import argparse, json, os, platform, time
import numpy as np, torch


def cpu_name():
    try:
        for l in open("/proc/cpuinfo"):
            if l.startswith("model name"): return l.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor()


def bench(fn, reps):
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts)), float(np.min(ts))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--onnx_dir", default="onnx")
    ap.add_argument("--student", default="", help="optional: also time PyTorch fp32")
    ap.add_argument("--threads", type=int, default=os.cpu_count())
    ap.add_argument("--reps", type=int, default=7)
    ap.add_argument("--out", default="assets/bench_cpu_v3.json")
    a = ap.parse_args()
    torch.set_num_threads(a.threads)
    import onnxruntime as ort
    from export_v3 import ort_session, ort_sample, torch_sample
    cap = "a red smiley face with 'HELLO WORLD' next to it, face on the right"
    d = a.onnx_dir; rows = []
    combos = [("ORT int8 U-Net + fp32 text encoder", "text_encoder.onnx", "unet_distilled_int8.onnx"),
              ("ORT int8 U-Net + int8 text encoder", "text_encoder_int8.onnx", "unet_distilled_int8.onnx"),
              ("ORT fp32", "text_encoder.onnx", "unet_distilled.onnx")]
    for name, tf, uf in combos:
        if not (os.path.exists(os.path.join(d, tf)) and os.path.exists(os.path.join(d, uf))): continue
        te, un = ort_session(os.path.join(d, tf), a.threads), ort_session(os.path.join(d, uf), a.threads)
        for steps in (2, 4):
            med, mn = bench(lambda: ort_sample(te, un, cap, steps, 0), a.reps)
            rows.append(dict(path=name, steps=steps, median_s=med, min_s=mn))
            print(f"{name:40s} {steps} steps: median {med:.3f}s (min {mn:.3f}s)", flush=True)
    if a.student:
        from distill_v3 import load_student
        s = load_student(a.student, "cpu")
        for steps in (2, 4):
            med, mn = bench(lambda: torch_sample(s, cap, steps, 0), a.reps)
            rows.append(dict(path="PyTorch fp32", steps=steps, median_s=med, min_s=mn))
    rep = dict(cpu=cpu_name(), cores=os.cpu_count(), threads=a.threads, reps=a.reps, torch=torch.__version__,
               onnxruntime=ort.__version__, onnx_dir=os.path.abspath(d), rows=rows)
    os.makedirs(os.path.dirname(a.out) or ".", exist_ok=True)
    json.dump(rep, open(a.out, "w"), indent=1); print(json.dumps(rep, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/grinpress_v3/run_v3.sh
#!/usr/bin/env bash
# grinpress v3: idempotent, unattended end-to-end run (Colab A100). Safe to start again at any time: every stage
# is skipped when its marker exists, and resumes from its newest checkpoint / chunk otherwise. DONE short-circuits.
#   stages: v1 teacher download -> CRNN train (stage 0, GPU) -> pregen teacher data -> teacher fine-tune (Phase 3)
#           -> reflow pairs (Phase 4a) -> student distill (Phase 4b) -> export + eval + bench (Phase 5) -> DONE
# Every file written to Drive is < 100 MB (sharded ckpts, 1000-pair shards, fp16 weights, split fp32 ONNX).
set -uo pipefail
cd "$(dirname "$0")"
export PYTORCH_CUDA_ALLOC_CONF=${PYTORCH_CUDA_ALLOC_CONF:-expandable_segments:True}
DRIVE=${DRIVE:-/content/drive/MyDrive/grinpress_v3}; export DRIVE
LOCAL=${LOCAL:-/content/grinpress_v3_local}
# ---- Phase 2 stage 0 (CRNN on the GPU)
C_NTRAIN=${C_NTRAIN:-120000}; C_EPOCHS=${C_EPOCHS:-30}; C_BS=${C_BS:-256}; C_LR=${C_LR:-2e-3}; C_HARD=${C_HARD:-0.4}
C_WIDTH=${C_WIDTH:-"32 64 128 192 256"}; C_SEQ=${C_SEQ:-4}; C_SEED=${C_SEED:-31}
CRNN_FROM_REPO=${CRNN_FROM_REPO:-0}; REQUIRE_CRNN_GATE=${REQUIRE_CRNN_GATE:-1}
# ---- Phase 3 (teacher)
PREGEN_N=${PREGEN_N:-100000}
T_BATCH=${T_BATCH:-128}; T_MICRO=${T_MICRO:-64}; T_STEPS=${T_STEPS:-40000}; T_LR=${T_LR:-1e-4}
T_VAL_EVERY=${T_VAL_EVERY:-1000}; T_STOP=${T_STOP:-0.95}; T_VAL_STEPS=${T_VAL_STEPS:-50}; VAL_LIMIT=${VAL_LIMIT:-0}
REQUIRE_TEACHER_GATE=${REQUIRE_TEACHER_GATE:-0}   # 1 = stop the run if the teacher never beats T_STOP on the 48-grid
# ---- Phase 4 (pairs + student)
NUM_PAIRS=${NUM_PAIRS:-50000}; SHARD=${SHARD:-1000}; GEN_BATCH=${GEN_BATCH:-128}; TEACHER_STEPS=${TEACHER_STEPS:-50}
S_BATCH=${S_BATCH:-128}; S_MICRO=${S_MICRO:-32}; S_STEPS=${S_STEPS:-20000}; S_LR=${S_LR:-5e-5}; S_VAL_EVERY=${S_VAL_EVERY:-2000}
EVAL_TEACHER_STEPS=${EVAL_TEACHER_STEPS:-50}; BENCH_REPS=${BENCH_REPS:-5}
MAX_RETRIES=${MAX_RETRIES:-5}
WORKERS=${WORKERS:-$(( $(nproc) - 1 ))}
mkdir -p "$DRIVE"/{teacher,teacher_ckpt,pairs_v3,student,student_ckpt,onnx,assets} "$LOCAL"/{teacher_data,teacher_ckpt,pairs,student_ckpt,export}
log() { echo "[run_v3 $(date '+%F %T')] $*"; }
beat() { python - "$@" <<PY || true
import json, sys, time, os
p = "$DRIVE/status.json"; d = {"phase": sys.argv[1], "time": time.strftime("%F %T"), "unix": int(time.time())}
if len(sys.argv) > 2: d["note"] = " ".join(sys.argv[2:])
open(p + ".tmp", "w").write(json.dumps(d)); os.replace(p + ".tmp", p)
PY
}
wait_drive() { for i in $(seq 1 60); do ls "$DRIVE" >/dev/null 2>&1 && return 0; log "Drive not reachable, waiting ($i/60)"; sleep 15; done; return 1; }
copy_to_drive() { cp "$1" "$2.tmp" && mv "$2.tmp" "$2"; }
if [ -f "$DRIVE/DONE" ]; then log "DONE marker present, nothing to do"; exit 0; fi

# ---------------------------------------------------------------- 0) inputs
if [ ! -f "$LOCAL/v1_teacher.safetensors" ]; then
  beat download_v1_teacher
  python -c "from huggingface_hub import hf_hub_download; import shutil; shutil.copy(hf_hub_download('skyuu72/grinpress','model.safetensors'), '$LOCAL/v1_teacher.safetensors')" || { beat error teacher_download; exit 1; }
fi

# ---------------------------------------------------------------- 0b) Phase 2 CRNN (stage 0, GPU; resumable via Drive ckpt)
# Trains the improved recognizer (scale channel, wider model, hard-class data) unless CRNN_DONE exists on Drive, or a
# CRNN that already passed the >=98% ref48 gate was committed (CRNN_FROM_REPO=1 skips training and uses artifacts/).
mkdir -p "$DRIVE/crnn" "$LOCAL/crnn_cache"
if [ "$CRNN_FROM_REPO" = "1" ]; then
  [ -f artifacts/crnn.safetensors ] || { log "artifacts/crnn.safetensors missing"; beat error no_crnn; exit 1; }
  [ -f "$DRIVE/CRNN_DONE" ] || { copy_to_drive artifacts/crnn.safetensors "$DRIVE/assets/crnn.safetensors"; echo '{"source":"repo"}' > "$DRIVE/CRNN_DONE"; }
fi
stuck=0
while [ ! -f "$DRIVE/CRNN_DONE" ]; do
  beat crnn
  python train_crnn.py --n_train $C_NTRAIN --n_val 2000 --epochs $C_EPOCHS --bs $C_BS --lr $C_LR --in_ch 2 --hard_frac $C_HARD \
    --width $C_WIDTH --seq_layers $C_SEQ --seed $C_SEED --workers $WORKERS --cache_dir "$LOCAL/crnn_cache" \
    --resume_ckpt "$DRIVE/crnn/crnn_ckpt.pt" --out "$LOCAL/crnn.safetensors" --report "$LOCAL/crnn_train.json" && {
      cp "$LOCAL/crnn.safetensors" artifacts/crnn.safetensors
      copy_to_drive "$LOCAL/crnn.safetensors" "$DRIVE/assets/crnn.safetensors"
      copy_to_drive "$LOCAL/crnn_train.json" "$DRIVE/assets/crnn_train.json"
      python -c "import json; r=json.load(open('$LOCAL/crnn_train.json')); json.dump(dict(source='stage0', ref48=r['final_ref48'], ref48_gtbox=r['final_ref48_with_gt_boxes'], val=r['final_val'], gate=r['gate_ref48_exact_ge_0_98']), open('$DRIVE/CRNN_DONE.tmp','w'))" \
        && mv "$DRIVE/CRNN_DONE.tmp" "$DRIVE/CRNN_DONE"; break; }
  wait_drive || { log "Drive gone 15 min; exiting for remount"; exit 1; }
  stuck=$((stuck + 1)); log "crnn training failed ($stuck/$MAX_RETRIES); resuming from Drive ckpt"
  [ "$stuck" -ge "$MAX_RETRIES" ] && { beat error crnn_stuck; exit 1; }
  sleep 20
done
[ -f "$DRIVE/assets/crnn.safetensors" ] && cp "$DRIVE/assets/crnn.safetensors" artifacts/crnn.safetensors
[ -f artifacts/crnn.safetensors ] || { log "no CRNN"; beat error no_crnn; exit 1; }
CGATE=$(python -c "import json; print(json.load(open('$DRIVE/CRNN_DONE')).get('gate', 'n/a'))")
log "CRNN stage 0 done; ref48 >= 0.98 gate: $CGATE"
if [ "$REQUIRE_CRNN_GATE" = "1" ] && [ "$CGATE" != "True" ] && [ "$CRNN_FROM_REPO" != "1" ]; then beat error crnn_gate_failed; exit 1; fi

# ---------------------------------------------------------------- 1) Phase 3 teacher fine-tune
if [ ! -f "$DRIVE/TEACHER_DONE" ]; then
  beat pregen
  stuck=0
  until python train_teacher_v3.py pregen --data_dir "$LOCAL/teacher_data" --num_samples $PREGEN_N --workers $WORKERS --heartbeat "$DRIVE/status.json"; do
    stuck=$((stuck + 1)); log "pregen failed ($stuck/$MAX_RETRIES)"; [ $stuck -ge $MAX_RETRIES ] && { beat error pregen; exit 1; }; sleep 10
  done
  stuck=0
  while [ ! -f "$DRIVE/TEACHER_DONE" ]; do
    before=$(ls "$LOCAL"/teacher_ckpt/ckpt_*.pt "$DRIVE"/teacher_ckpt/*.manifest.json 2>/dev/null | tail -2 | tr '\n' ' ')
    python train_teacher_v3.py train --init "$LOCAL/v1_teacher.safetensors" --crnn artifacts/crnn.safetensors \
      --data_dir "$LOCAL/teacher_data" --checkpoint_dir "$LOCAL/teacher_ckpt" --checkpoint_mirror "$DRIVE/teacher_ckpt" \
      --out_dir "$DRIVE/teacher" --batch_size $T_BATCH --micro_batch $T_MICRO --lr $T_LR --max_steps $T_STEPS \
      --val_every $T_VAL_EVERY --val_steps $T_VAL_STEPS --val_limit $VAL_LIMIT --stop_exact $T_STOP --heartbeat "$DRIVE/status.json" --done_marker "$DRIVE/TEACHER_DONE" && break
    wait_drive || { log "Drive gone 15 min; exiting so the notebook monitor can remount + relaunch"; exit 1; }
    after=$(ls "$LOCAL"/teacher_ckpt/ckpt_*.pt "$DRIVE"/teacher_ckpt/*.manifest.json 2>/dev/null | tail -2 | tr '\n' ' ')
    if [ "$after" != "$before" ]; then stuck=0; log "teacher died after progress; resuming";
    elif [ "$T_MICRO" -gt 1 ]; then T_MICRO=$((T_MICRO / 2)); stuck=0; log "teacher failed, no progress -> T_MICRO=$T_MICRO";
    else stuck=$((stuck + 1)); log "teacher failed at micro 1 ($stuck/$MAX_RETRIES)"; fi
    [ "$stuck" -ge "$MAX_RETRIES" ] && { beat error teacher_stuck; exit 1; }
    beat teacher_retry "T_MICRO=$T_MICRO"; sleep 20
  done
fi
TEACHER="$DRIVE/teacher/teacher_v3_fp16.safetensors"
[ -f "$TEACHER" ] || { log "missing $TEACHER"; beat error no_teacher; exit 1; }
GATE=$(python -c "import json; print(json.load(open('$DRIVE/TEACHER_DONE')).get('best_grid48_exact', 0))")
log "teacher best 48-grid CRNN exact = $GATE (target > $T_STOP)"
if [ "$REQUIRE_TEACHER_GATE" = "1" ] && python -c "import sys; sys.exit(0 if float('$GATE') > float('$T_STOP') else 1)"; then :;
elif [ "$REQUIRE_TEACHER_GATE" = "1" ]; then beat error teacher_gate_failed "best=$GATE"; exit 1; fi

# ---------------------------------------------------------------- 2) Phase 4a reflow pairs (Drive shards, resumable)
stuck=0
while true; do
  before=$(ls "$DRIVE"/pairs_v3/pairs_*.pt "$DRIVE"/pairs_v3/partial_*/chunk_*.pt 2>/dev/null | wc -l)
  python distill_v3.py gen_pairs --teacher "$TEACHER" --out_dir "$DRIVE/pairs_v3" --num_pairs $NUM_PAIRS --shard_size $SHARD \
    --steps $TEACHER_STEPS --cfg 5.0 --batch_size $GEN_BATCH --heartbeat "$DRIVE/status.json" && break
  wait_drive || { log "Drive gone 15 min; exiting for remount"; exit 1; }
  after=$(ls "$DRIVE"/pairs_v3/pairs_*.pt "$DRIVE"/pairs_v3/partial_*/chunk_*.pt 2>/dev/null | wc -l)
  if [ "$after" != "$before" ]; then stuck=0; log "gen_pairs died after progress; resuming";
  elif [ "$GEN_BATCH" -gt 8 ]; then GEN_BATCH=$((GEN_BATCH / 2)); stuck=0; log "gen_pairs failed, no progress -> GEN_BATCH=$GEN_BATCH";
  else stuck=$((stuck + 1)); log "gen_pairs failed at min batch ($stuck/$MAX_RETRIES)"; fi
  [ "$stuck" -ge "$MAX_RETRIES" ] && { beat error gen_pairs_stuck; exit 1; }
  beat gen_pairs_retry "GEN_BATCH=$GEN_BATCH"; sleep 20
done
NS=$(ls "$DRIVE"/pairs_v3/pairs_*.pt 2>/dev/null | wc -l)
[ "$NS" -ge $(( (NUM_PAIRS + SHARD - 1) / SHARD )) ] || { log "pairs incomplete ($NS)"; beat error pairs_incomplete; exit 1; }

# ---------------------------------------------------------------- 3) Phase 4b student distill
if [ ! -f "$DRIVE/STUDENT_DONE" ]; then
  beat copy_pairs
  for f in "$DRIVE"/pairs_v3/pairs_*.pt; do
    b=$(basename "$f")
    if [ ! -f "$LOCAL/pairs/$b" ] || [ "$(stat -c%s "$f")" != "$(stat -c%s "$LOCAL/pairs/$b")" ]; then cp "$f" "$LOCAL/pairs/$b.tmp" && mv "$LOCAL/pairs/$b.tmp" "$LOCAL/pairs/$b"; fi
  done
  stuck=0
  while [ ! -f "$DRIVE/STUDENT_DONE" ]; do
    before=$(ls "$LOCAL"/student_ckpt/ckpt_*.pt "$DRIVE"/student_ckpt/*.manifest.json 2>/dev/null | tail -2 | tr '\n' ' ')
    python distill_v3.py train --teacher "$TEACHER" --crnn artifacts/crnn.safetensors --pair_dir "$LOCAL/pairs" \
      --checkpoint_dir "$LOCAL/student_ckpt" --checkpoint_mirror "$DRIVE/student_ckpt" --out_dir "$DRIVE/student" \
      --batch_size $S_BATCH --micro_batch $S_MICRO --lr $S_LR --max_steps $S_STEPS --val_every $S_VAL_EVERY --val_limit $VAL_LIMIT \
      --heartbeat "$DRIVE/status.json" --done_marker "$DRIVE/STUDENT_DONE" && break
    wait_drive || { log "Drive gone 15 min; exiting for remount"; exit 1; }
    after=$(ls "$LOCAL"/student_ckpt/ckpt_*.pt "$DRIVE"/student_ckpt/*.manifest.json 2>/dev/null | tail -2 | tr '\n' ' ')
    if [ "$after" != "$before" ]; then stuck=0; log "student died after progress; resuming";
    elif [ "$S_MICRO" -gt 1 ]; then S_MICRO=$((S_MICRO / 2)); stuck=0; log "student failed, no progress -> S_MICRO=$S_MICRO";
    else stuck=$((stuck + 1)); log "student failed at micro 1 ($stuck/$MAX_RETRIES)"; fi
    [ "$stuck" -ge "$MAX_RETRIES" ] && { beat error student_stuck; exit 1; }
    beat student_retry "S_MICRO=$S_MICRO"; sleep 20
  done
fi
STUDENT="$DRIVE/student/student_ema_fp16.safetensors"
[ -f "$STUDENT" ] || { log "missing $STUDENT"; beat error no_student; exit 1; }
[ -f "$DRIVE/TRAIN_DONE" ] || { python -c "import json,time; json.dump(dict(time=time.strftime('%F %T'), crnn=json.load(open('$DRIVE/CRNN_DONE')), teacher=json.load(open('$DRIVE/TEACHER_DONE')), student='$STUDENT'), open('$DRIVE/TRAIN_DONE.tmp','w'), indent=1)" && mv "$DRIVE/TRAIN_DONE.tmp" "$DRIVE/TRAIN_DONE"; }

# ---------------------------------------------------------------- 4) Phase 5 export + eval + bench
beat export
python export_v3.py --student "$STUDENT" --out_dir "$LOCAL/export" --report "$LOCAL/export/export_report.json" --split_big || { beat error export; exit 1; }
for f in text_encoder.onnx text_encoder_int8.onnx unet_distilled_int8.onnx student_ema_fp16.safetensors export_report.json torch_ortfp32_ortint8.png; do
  copy_to_drive "$LOCAL/export/$f" "$DRIVE/onnx/$f"
done
for f in "$LOCAL"/export/unet_distilled.onnx.part* "$LOCAL"/export/unet_distilled.onnx.manifest.json; do
  [ -e "$f" ] && copy_to_drive "$f" "$DRIVE/onnx/$(basename "$f")"
done
beat eval
python eval_ocr.py models --unet "$LOCAL/export/unet_distilled_int8.onnx" --text "$LOCAL/export/text_encoder_int8.onnx" \
  --teacher "$TEACHER" --teacher_steps $EVAL_TEACHER_STEPS --assets "$LOCAL/assets" --tess v3 || { beat error eval; exit 1; }
for f in "$LOCAL"/assets/*; do copy_to_drive "$f" "$DRIVE/assets/$(basename "$f")"; done
python bench_cpu.py --onnx_dir "$LOCAL/export" --out "$DRIVE/assets/bench_cpu_colab_NOT_TARGET_CPU.json" --reps $BENCH_REPS || true
# ---------------------------------------------------------------- 5) size check + optional HF push (token only from env)
beat hf_push
python - <<'PY' || true
import os, json, time
D = os.environ["DRIVE"]; big = []
for sub in ("onnx", "student", "teacher", "assets", "crnn", "teacher_ckpt", "student_ckpt"):
    p = os.path.join(D, sub)
    for f in (os.listdir(p) if os.path.isdir(p) else []):
        fp = os.path.join(p, f)
        if os.path.isfile(fp) and os.path.getsize(fp) >= 100 * 2**20: big.append((f"{sub}/{f}", os.path.getsize(fp)))
print("[size check] files >= 100 MB on Drive:", big or "none")
res = {"time": time.strftime("%F %T"), "pushed": False}
tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
repo = os.environ.get("HF_REPO", "skyuu72/grinpress-v3")
if not tok:
    res["reason"] = "no HF_TOKEN in env; skipped"
else:
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=tok)
        for sub in ("onnx", "student", "assets"):
            if os.path.isdir(os.path.join(D, sub)):
                api.upload_folder(repo_id=repo, repo_type="model", folder_path=os.path.join(D, sub), path_in_repo=sub,
                                  allow_patterns=["*.onnx", "*.onnx.part*", "*.manifest.json", "*.safetensors", "*.json", "*.png"],
                                  commit_message=f"grinpress v3 Colab run: {sub}")
        for f in ("TRAIN_DONE", "CRNN_DONE", "TEACHER_DONE", "teacher/teacher_val_history.json"):
            if os.path.exists(os.path.join(D, f)):
                api.upload_file(path_or_fileobj=os.path.join(D, f), path_in_repo="run/" + os.path.basename(f), repo_id=repo, repo_type="model")
        res.update(pushed=True, repo=repo)
    except Exception as e:  # never fatal; message only (never the token)
        res["reason"] = f"{type(e).__name__}: {str(e)[:300]}"
print("[hf push]", res)
json.dump(res, open(os.path.join(D, "HF_PUSH.json"), "w"), indent=1)
PY
python - <<PY
import json, time
s = json.load(open("$LOCAL/assets/eval_ocr_v3.json"))["summary"]
td = json.load(open("$DRIVE/TEACHER_DONE"))
json.dump({"finished": time.strftime("%F %T"), "teacher_best_grid48_exact": td.get("best_grid48_exact"),
           "teacher_gate_passed": (td.get("best_grid48_exact") or 0) > float("$T_STOP"),
           "student_grid48": {k: v.get("crnn") for k, v in s.items()}, "export": "onnx/", "assets": "assets/",
           "hf_push": json.load(open("$DRIVE/HF_PUSH.json")) if __import__("os").path.exists("$DRIVE/HF_PUSH.json") else None},
          open("$DRIVE/DONE.tmp", "w"), indent=1)
PY
mv "$DRIVE/DONE.tmp" "$DRIVE/DONE"
beat done
log "all done :3"


In [ ]:
# Phase-2 CRNN recognizer weights (frozen OCR loss + evaluator), embedded so no credentials are needed
import base64, hashlib
B64 = (
    '...'
)
data = base64.b64decode(B64)
assert hashlib.sha256(data).hexdigest() == '88529075f64fcd0e2209e9e3cdf29d860ef7106c9fad62afa3e61ec1af45317d', 'CRNN weights corrupted'
open('/content/grinpress_v3/artifacts/crnn.safetensors', 'wb').write(data); print('crnn.safetensors', len(data), 'bytes OK')

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q safetensors onnx onnxruntime onnxscript opencv-python-headless pytesseract huggingface_hub
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.is_bf16_supported() if torch.cuda.is_available() else '')

## Launch (background, idempotent)
Defaults: 100k pregen samples; teacher batch 128 (micro 64), up to 40k steps, validated every 1000 steps, early stop when the CRNN reads > 95% of the 48-grid exactly; 50k pairs (50 Euler steps, CFG 5, GEN_BATCH 128); student batch 128 (micro 32), 20k steps. Micro-batches / GEN_BATCH halve themselves on CUDA OOM. Stage 0 trains the CRNN first (120k samples, 30 epochs, ~0.5 h; REQUIRE_CRNN_GATE=1 stops the run if it reads < 98% of the 48 refs). Optional: add a Colab Secret `HF_TOKEN` to push the final artifacts to the private HF repo skyuu72/grinpress-v3. Estimated wall time on an A100 40GB: ~0.5 h CRNN + ~0.3 h pregen + 1-3 h teacher + ~2 h pairs + ~1-1.5 h student + ~0.3 h export/eval ≈ 5-7 h.

In [ ]:
import os, subprocess
RUN = dict(PREGEN_N='100000', T_BATCH='128', T_MICRO='64', T_STEPS='40000', T_LR='1e-4', T_VAL_EVERY='1000', T_STOP='0.95',
           REQUIRE_CRNN_GATE='1', CRNN_FROM_REPO='0', REQUIRE_TEACHER_GATE='0', NUM_PAIRS='50000', SHARD='1000', GEN_BATCH='128', S_BATCH='128', S_MICRO='32',
           S_STEPS='20000', S_LR='5e-5')
DRIVE = '/content/drive/MyDrive/grinpress_v3'; LOG = f'{DRIVE}/run.log'

def running():
    return subprocess.run(['pgrep', '-f', 'bash /content/grinpress_v3/run_v3.sh'], capture_output=True).returncode == 0

def launch():
    if os.path.exists(f'{DRIVE}/DONE'):
        print('DONE marker found: the run is already finished 🐾'); return
    if running():
        print('run_v3.sh already running, not starting a second copy'); return
    env = dict(os.environ, DRIVE=DRIVE, LOCAL='/content/grinpress_v3_local', **RUN)
    try:  # optional: HF token from Colab Secrets (key icon, name HF_TOKEN, notebook access on); never stored in the notebook
        from google.colab import userdata
        t = userdata.get('HF_TOKEN')
        if t: env['HF_TOKEN'] = t
    except Exception as e:
        print('no HF_TOKEN secret -> final HF push will be skipped (', type(e).__name__, ')')
    print('HF push at the end:', 'enabled' if env.get('HF_TOKEN') else 'skipped (no token)')
    subprocess.Popen(f'nohup setsid bash /content/grinpress_v3/run_v3.sh >> "{LOG}" 2>&1 &', shell=True, env=env, cwd='/content/grinpress_v3')
    print('launched in background; log ->', LOG)

launch()

## Monitor until done (safe to interrupt or re-run; it relaunches the run if the process died)

In [ ]:
import time, json, os, subprocess, threading
from IPython.display import clear_output
from google.colab import drive

def drive_ok(timeout=12):
    """listdir can hang forever on a wedged Drive FUSE; bound it with a short thread join."""
    box = []
    def _ls():
        try:
            os.listdir(DRIVE); box.append(True)
        except OSError:
            box.append(False)
    th = threading.Thread(target=_ls, daemon=True); th.start(); th.join(timeout)
    if th.is_alive():
        print('drive_ok: listdir hung >%ss (FUSE wedged?)' % timeout); return False
    return bool(box and box[0])

def remount():
    for i in range(6):
        try:
            drive.mount('/content/drive', force_remount=True, timeout_ms=180000)
            if drive_ok(): print('Drive remounted 🐾'); return True
        except Exception as e:
            print(f'remount attempt {i+1} failed: {e}')
        time.sleep(15 * (i + 1))
    return False

relaunches, bad = 0, 0
while True:
    try:
        if os.path.exists(f'{DRIVE}/DONE'): break
        if not drive_ok():
            bad += 1; print(time.strftime('%H:%M:%S'), f'Drive not reachable ({bad})')
            if bad >= 3: remount(); bad = 0
            time.sleep(60); continue
        bad = 0
        try: st = open(f'{DRIVE}/status.json').read()
        except OSError: st = '(no heartbeat yet)'
        try: tail = subprocess.run(['tail', '-n', '15', LOG], capture_output=True, text=True, timeout=60).stdout
        except Exception: tail = ''
        clear_output(wait=True)
        print(time.strftime('%H:%M:%S'), 'status:', st); print(tail)
        if not running() and not os.path.exists(f'{DRIVE}/DONE'):
            if relaunches >= 20:
                print('run keeps dying; check run.log 🐾'); break
            relaunches += 1; print(f'process not running -> relaunch #{relaunches}'); launch()
    except Exception as e:  # the monitor itself must never die
        print('monitor error:', e)
    time.sleep(60)
if os.path.exists(f'{DRIVE}/DONE'):
    print('DONE :3', open(f'{DRIVE}/DONE').read())

## Results

In [ ]:
from IPython.display import Image, display
import json, os
for f in ('samples_4step_v3.png', 'samples_2step_v3.png', 'teacher_vs_student_v3.png'):
    p = f'{DRIVE}/assets/{f}'
    if os.path.exists(p): print(f); display(Image(p))
for f in ('DONE', 'teacher/teacher_val_history.json', 'assets/eval_ocr_v3.json', 'onnx/export_report.json'):
    p = f'{DRIVE}/{f}'
    if os.path.exists(p): print(f, open(p).read()[:3000])